# Sentiment Analyzer — YouTube Live Stream Chat Analysis

**Objective:** Aspect-based sentiment analysis of Portuguese-language YouTube live stream chat using open-source LLMs via vLLM with structured output guarantees.

**Data Source:** CazéTV — RD Congo vs Jamaica (FIFA 2026 World Cup Playoff Final) — 12,711 chat messages.

**Experimental Design:** 3 models × 3 prompting strategies × 5 repetitions = 45 classification runs.

| Model | Family | Params | Purpose |
|---|---|---|---|
| Qwen/Qwen3.5-4B | Qwen | 4B | Strong multilingual, Apache 2.0 |
| microsoft/Phi-4-mini-instruct | Phi-4 | 3.8B | Strong instruction following, float16 compatible |
| deepseek-ai/DeepSeek-R1-Distill-Qwen-7B | DeepSeek | 7B | Reasoning-distilled |

| Strategy | Description |
|---|---|
| Zero-shot | No examples — model’s pretrained understanding |
| ICL | 6 labeled examples (2 per class) from 18-message holdout pool |
| Chain-of-Thought | Step-by-step reasoning before classification |

**Classification Dimensions:**
- **Sentiment:** Positive / Negative / Neutral
- **Aspect:** content / presenter / community / audio_video / meta

**Ground Truth:** Dual-tier — 200 human-annotated (Gold, 182 after ICL holdout) + ~300 consensus proxy (Silver, ≥2/3 model agreement).

In [ ]:
# Phase 3 — Cell 3.1: Install dependencies (pinned versions)
#
# Colab: mounts Drive early (idempotent) so the restart sentinel can be stored
# on a path that survives kernel restarts (/tmp is wiped on every restart).
# Installs vLLM first (transitive deps may downgrade numpy), then force-reinstalls
# numpy 2.4.4 + scipy last. Tests the loaded C extension and SIGKILL-restarts once
# if stale. Drive-mounted sentinel prevents infinite restart loops.
#
# Local Docker: verifies pre-installed image packages.
# Time: O(P) where P = number of packages. Space: O(1).

import os, importlib
from pathlib import Path as _Path

_local_mode = bool(os.environ.get('SENTIMENT_DATA_ROOT'))

if not _local_mode:
    from google.colab import drive as _drive
    _drive.mount('/content/drive', force_remount=False)
    _sentinel_dir = _Path('/content/drive/MyDrive/sentiment_analyzer')
    _sentinel_dir.mkdir(parents=True, exist_ok=True)
    _restart_sentinel = _sentinel_dir / '.numpy_restart_done'

    !pip install -q vllm==0.18.1 pydantic==2.12.3 2>/dev/null
    !pip install -q polars==1.39.3 pandas==3.0.2 pyarrow==23.0.1 2>/dev/null
    !pip install -q scikit-learn==1.6.1 pingouin==0.6.1 statsmodels==0.14.4 2>/dev/null
    !pip install -q plotly==6.6.0 seaborn==0.13.2 matplotlib==3.10.8 kaleido==1.2.0 2>/dev/null
    !pip install -q tqdm==4.67.3 nest-asyncio==1.6.0 emoji==2.14.1 2>/dev/null
    !pip install -q --force-reinstall numpy==2.4.4 scipy==1.17.1 2>/dev/null
    print('All pip installs done.')

    try:
        from numpy._core.umath import _center  # noqa: F401
        import numpy as _np, vllm as _vllm
        print(f'Colab: runtime OK \u2014 numpy {_np.__version__}, vllm {_vllm.__version__}')
        _restart_sentinel.unlink(missing_ok=True)
    except ImportError:
        if _restart_sentinel.exists():
            raise RuntimeError(
                'numpy C extension still stale after restart \u2014 pip version conflict detected.\n'
                'Diagnose with: !pip show numpy scipy vllm'
            )
        _restart_sentinel.write_text('1')
        print('\u26a0\ufe0f  numpy C extension stale (old .so in memory) \u2014 restarting runtime.')
        print('   Colab will reconnect; click \'Run all\' to continue.')
        import os as _os; _os.kill(_os.getpid(), 9)
else:
    print('Local Docker: packages pre-installed in image\n')
    checks = [
        ('polars',      'polars'),
        ('pandas',      'pandas'),
        ('pyarrow',     'pyarrow'),
        ('numpy',       'numpy'),
        ('scipy',       'scipy'),
        ('sklearn',     'scikit-learn'),
        ('pingouin',    'pingouin'),
        ('statsmodels', 'statsmodels'),
        ('plotly',      'plotly'),
        ('seaborn',     'seaborn'),
        ('matplotlib',  'matplotlib'),
        ('tqdm',        'tqdm'),
        ('emoji',       'emoji'),
        ('pydantic',    'pydantic'),
        ('requests',    'requests'),
        ('nest_asyncio','nest-asyncio'),
    ]
    all_ok = True
    for mod, pkg in checks:
        try:
            m = importlib.import_module(mod)
            ver = getattr(m, '__version__', '?')
            print(f'  OK  {pkg:<18} {ver}')
        except ImportError:
            print(f'  MISSING  {pkg}')
            all_ok = False
    print()
    print('All packages OK' if all_ok else 'Some packages missing \u2014 rebuild image')


In [ ]:
# Phase 3 — Cell 3.2: Environment detection, imports, Drive mount, HF token
#
# Detects Colab vs local Docker, mounts Drive, sets HF_HOME to persistent
# cache on Drive, and configures vLLM spawn method before any CUDA import.
# Time: O(1). Space: O(1).

import os
import nest_asyncio
nest_asyncio.apply()

import polars as pl
pl.Config.set_tbl_rows(20)

try:
    from IPython import get_ipython as _gip
    IN_COLAB = 'google.colab' in str(_gip())
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    DATA_ROOT = __import__('pathlib').Path('/content/drive/MyDrive/sentiment_analyzer')
    HF_TOKEN  = userdata.get('HF_TOKEN')
    USE_VLLM  = True

    _hf_cache = DATA_ROOT / 'hf_cache'
    _hf_cache.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(_hf_cache)
    print(f'HF cache: {_hf_cache}')

    os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

    !huggingface-cli login --token $HF_TOKEN
else:
    from pathlib import Path
    DATA_ROOT  = Path(os.environ.get('SENTIMENT_DATA_ROOT', './data'))
    HF_TOKEN   = os.environ.get('HF_TOKEN', '')
    USE_VLLM   = False
    OLLAMA_URL = os.environ.get('OLLAMA_URL', 'http://ollama:11434')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Local Docker'}")
print(f'Data root:   {DATA_ROOT}')
print(f"Backend:     {'vLLM (GPU)' if USE_VLLM else 'Ollama (CPU)'}")


In [18]:
# Phase 3 — Cell 3.3: Load raw data from Drive / local mount

import json
from pathlib import Path

base_path = DATA_ROOT / "raw_data"

videos: list[dict] = []
for folder in sorted(base_path.iterdir()):
    meta_path = folder / "metadata.json"
    chat_path = folder / "chat_messages.csv"
    if not (folder.is_dir() and meta_path.exists() and chat_path.exists()):
        continue
    meta = json.loads(meta_path.read_text())
    raw  = pl.read_csv(str(chat_path))
    videos.append({"video_id": folder.name, "metadata": meta, "raw_chat": raw})

assert videos, f"No usable video folders found under {base_path}"

print(f"{'video_id':<16} {'msgs':>6}  {'duration':>8}  title")
print("-" * 80)
for v in videos:
    m = v["metadata"]
    dur_h = m.get("duration_seconds", 0) / 3600
    title = m.get("title", "")[:45]
    print(f"{v['video_id']:<16} {v['raw_chat'].shape[0]:>6,}  {dur_h:>7.1f}h  {title}")

print(f"\nTotal: {len(videos)} videos  {sum(v['raw_chat'].shape[0] for v in videos):,} raw messages")


video_id           msgs  duration  title
--------------------------------------------------------------------------------
51W1AoWLo2U       9,134      3.0h  JOGO COMPLETO: NOVA CALEDÔNIA X JAMAICA | REP
KLzlxfxKzec         397      1.5h  AO VIVO: SALAH DE SAÍDA DO LIVERPOOL | ARGENT
OO-Qhz8lN44         407      1.5h  AO VIVO: ITÁLIA NA REPESCAGEM DA COPA, AMISTO
ewYXmeND0Zs       5,768      2.1h  WOLFSBURG X OL LYONNES AO VIVO: JOGO DA CHAMP
fkGZsNYTugU         956      1.4h  AO VIVO: ITÁLIA X IRLANDA DO NORTE | BRASIL E
g_Tfodf9bmk      12,665      4.5h  JOGO COMPLETO: BOLÍVIA X SURINAME | REPESCAGE
wYzzWC8L33Q         866      2.1h  BELGRANO X ATLÉTICO RAFAELA - COPA ARGENTINA 
yfyhoO9qse4         724      2.8h  JOGO COMPLETO: NANTES X STRASBOURG | LIGUE 1 

Total: 8 videos  30,917 raw messages


In [ ]:
# Phase 3 — Cell 3.4: GPU detection and vLLM configuration
#
# Detects GPU via nvidia-smi subprocess (avoids CUDA context init in the
# parent process so vLLM can use fork instead of spawn — faster cold-start).
# Time: O(1). Space: O(1).

if USE_VLLM:
    import subprocess, re

    _smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        capture_output=True, text=True,
    )
    if _smi.returncode != 0:
        raise RuntimeError(
            'nvidia-smi failed \u2014 is a GPU runtime selected?\n'
            'Runtime > Change runtime type > T4 GPU'
        )
    _gpu_line = _smi.stdout.strip().splitlines()[0]
    GPU_NAME, _mem_str = [s.strip() for s in _gpu_line.split(',')]
    GPU_MEM_GB = int(_mem_str) / 1024
    IS_T4 = 'T4' in GPU_NAME

    VLLM_DTYPE    = 'half' if IS_T4 else 'auto'
    MAX_MODEL_LEN = 2048   if IS_T4 else 4096
    ENFORCE_EAGER = IS_T4
    GPU_MEM_UTIL  = 0.92   if IS_T4 else 0.90

    print(f'GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GiB)')
    print(f'vLLM config: dtype={VLLM_DTYPE}, max_model_len={MAX_MODEL_LEN}, '
          f'enforce_eager={ENFORCE_EAGER}, gpu_mem_util={GPU_MEM_UTIL}')
    print('Note: CUDA context NOT yet initialised \u2014 vLLM will use fork (faster).')
else:
    VLLM_DTYPE = MAX_MODEL_LEN = ENFORCE_EAGER = GPU_MEM_UTIL = None
    print('Local mode: CPU inference via Ollama \u2014 no GPU required')


---
## Phase 4: Dataset Structuring and Ground Truth

**Goal:** Clean raw chat data, sample 500 messages from volatility peaks, create hybrid Ground Truth.

- **Tier 1 (Gold):** 200 human-annotated messages (182 for evaluation after 18 ICL holdout)
- **Tier 2 (Silver):** ~300 consensus proxy messages (≥2/3 model agreement, generated after Phase 5)

In [20]:
# Phase 4 — Cell 4.1: Configuration

minimum_text_length = 5          # @param {type:"slider", min:1, max:100, step:1}
gt_per_video        = 50         # @param {type:"slider", min:20, max:200, step:10}
n_time_segments     = 5          # @param {type:"slider", min:3, max:10, step:1}
handle_emojis       = "convert_to_text"  # @param ["convert_to_text", "remove", "keep"]
volatility_bin_width = 60        # @param {type:"slider", min:10, max:300, step:10}
random_seed         = 42         # @param {type:"integer"}

BASE_DIR       = DATA_ROOT
CLEANED_DIR    = BASE_DIR / "cleaned_data"
GT_DIR         = BASE_DIR / "ground_truth"
CLASSIFIER_DIR = BASE_DIR / "classifier_output"
METRICS_DIR    = BASE_DIR / "metrics_output"
VIZ_DIR        = BASE_DIR / "visualizations"

for directory in [CLEANED_DIR, GT_DIR, CLASSIFIER_DIR, METRICS_DIR, VIZ_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

n_videos       = len(videos)
gt_human_size  = gt_per_video * n_videos               # total human annotation quota
gt_pool_size   = gt_human_size + (gt_per_video // 2) * n_videos  # human + consensus

print(f"Directories created under: {BASE_DIR}")
print(f"Videos: {n_videos}  |  gt_per_video: {gt_per_video}  |  segments: {n_time_segments}")
print(f"Total human GT target: {gt_human_size}  |  Full pool target: {gt_pool_size}")


Directories created under: /data
Videos: 8  |  gt_per_video: 50  |  segments: 5
Total human GT target: 400  |  Full pool target: 600


In [21]:
# Phase 4 — Cell 4.2: Text cleaning chain

import unicodedata
import re
from typing import Callable

TextCleaningStrategy = Callable[[pl.Expr], pl.Expr]

YOUTUBE_CHAT_SPAM_PATTERNS: tuple[str, ...] = (
    r"^![a-z]+\s*$",
    r"^(?:L+|W+|F+|GG)\s*$",
    r"^(?:\.+|\u2026+)\s*$",
    r"^https?://\S+\s*$",
    r"^\d+\s*$",
    r"^(?:kk)+\s*$",
)


def normalize_encoding(expr: pl.Expr) -> pl.Expr:
    """Normalize Unicode to NFC and fix common encoding artifacts.

    @param expr: Polars Utf8 expression.
    @returns: Normalized expression.
    Time: O(N * L). Space: O(N * L).
    """
    def _normalize_row(text: str) -> str:
        if text is None:
            return ""
        normalized = unicodedata.normalize("NFC", text)
        replacements = {
            "\u2018": "'", "\u2019": "'",
            "\u201c": '"', "\u201d": '"',
            "\u2013": "-", "\u2014": "-",
            "\u2026": "...", "\u00a0": " ", "\ufffd": "",
        }
        for old, new in replacements.items():
            normalized = normalized.replace(old, new)
        return normalized
    return expr.map_elements(_normalize_row, return_dtype=pl.Utf8)


def handle_emojis_fn(expr: pl.Expr, mode: str) -> pl.Expr:
    """Handle emoji characters based on the selected mode.

    @param expr: Polars Utf8 expression.
    @param mode: 'convert_to_text', 'remove', or 'keep'.
    @returns: Processed expression.
    Time: O(N * L). Space: O(N * L).
    """
    if mode == "keep":
        return expr
    if mode == "convert_to_text":
        import emoji
        def _convert(text: str) -> str:
            if text is None:
                return ""
            return emoji.demojize(text, delimiters=(" ", " "))
        return expr.map_elements(_convert, return_dtype=pl.Utf8)
    emoji_pattern = (
        r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
        r"\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251"
        r"\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF"
        r"\U00002600-\U000026FF\U0000FE00-\U0000FE0F\U0000200D]+"
    )
    return expr.str.replace_all(emoji_pattern, "")


def compose_cleaning_chain(strategies: list[TextCleaningStrategy]) -> TextCleaningStrategy:
    """Compose cleaning strategies left-to-right into a single function.

    @param strategies: Ordered list of text cleaning strategies.
    @returns: Composed strategy.
    Time: O(S). Space: O(1).
    """
    def _composed(expr: pl.Expr) -> pl.Expr:
        result = expr
        for strategy in strategies:
            result = strategy(result)
        return result
    return _composed


cleaning_chain = compose_cleaning_chain([
    lambda e: e.str.replace_all(r"<[^>]+>", ""),
    normalize_encoding,
    lambda e: handle_emojis_fn(e, handle_emojis),
    lambda e: e.str.replace_all(r"https?://\S+|www\.\S+", "[URL]"),
    lambda e: e.str.replace_all(r"@[\w.]+", "[USER]"),
    lambda e: e.map_elements(lambda s: re.sub(r'(.)\1{3,}', r'\1\1\1', s) if s is not None else s, return_dtype=pl.Utf8),
    lambda e: pl.when(e == e.str.to_uppercase()).then(e.str.to_lowercase()).otherwise(e),
    lambda e: e.str.replace_all(r"\t", " ").str.replace_all(r" {2,}", " ").str.strip_chars(),
])

spam_pattern = "|".join(f"({p})" for p in YOUTUBE_CHAT_SPAM_PATTERNS)

print("Cleaning chain composed with 8 strategies")

Cleaning chain composed with 8 strategies


In [22]:
%%time
# Phase 4 — Cell 4.3: Clean all videos — filter, clean, assign global row_id

all_cleaned: list[pl.DataFrame] = []
print(f"{'video_id':<16} {'before':>7}  {'after':>7}  {'survival':>9}")
print("-" * 50)

for v in videos:
    raw = v["raw_chat"]
    cleaned_v = (
        raw.lazy()
        .filter(pl.col("time_in_seconds") > 0.0)
        .filter(~pl.col("message").str.contains(r"^![a-z]+"))
        .with_columns(cleaning_chain(pl.col("message")).alias("message"))
        .filter(~pl.col("message").str.contains(spam_pattern))
        .filter(pl.col("message").str.len_chars() >= minimum_text_length)
        .collect()
        .with_row_index("local_row_id")
        .with_columns(pl.lit(v["video_id"]).alias("video_id"))
    )
    pct = cleaned_v.shape[0] / max(raw.shape[0], 1) * 100
    print(f"{v['video_id']:<16} {raw.shape[0]:>7,}  {cleaned_v.shape[0]:>7,}  {pct:>8.1f}%")
    all_cleaned.append(cleaned_v)

cleaned = pl.concat(all_cleaned).with_row_index("row_id", offset=0)
rows_before = sum(v["raw_chat"].shape[0] for v in videos)
rows_after  = cleaned.shape[0]

print("-" * 50)
print(f"{'TOTAL':<16} {rows_before:>7,}  {rows_after:>7,}  {rows_after/rows_before*100:>8.1f}%")

text_lengths = cleaned["message"].str.len_chars()
print(f"\nText length  mean={text_lengths.mean():.0f}  median={text_lengths.median()}  min={text_lengths.min()}  max={text_lengths.max()}")
assert rows_after >= gt_pool_size, f"Only {rows_after} cleaned messages available (need {gt_pool_size})"
cleaned.head(5)


video_id          before    after   survival
--------------------------------------------------
51W1AoWLo2U        9,134    8,457      92.6%
KLzlxfxKzec          397      371      93.5%
OO-Qhz8lN44          407      385      94.6%
ewYXmeND0Zs        5,768    5,015      86.9%
fkGZsNYTugU          956      911      95.3%
g_Tfodf9bmk       12,665   11,917      94.1%
wYzzWC8L33Q          866      792      91.5%
yfyhoO9qse4          724      685      94.6%
--------------------------------------------------
TOTAL             30,917   28,533      92.3%

Text length  mean=32  median=24.0  min=5  max=200
CPU times: user 875 ms, sys: 15.6 ms, total: 890 ms
Wall time: 902 ms


row_id,local_row_id,message_id,timestamp,time_in_seconds,author_hash,message,author_type,message_type,datetime,captured_at,video_id
u32,u32,str,str,f64,str,str,str,str,str,str,str
0,0,"""ChwKGkNJV2YtdF85dnBNREZRUFJsQW…","""0:02""",2.116,"""fa3d3bc6""","""fogo na babilonia""",null,"""text""","""2026-03-26T23:00:03.422591""","""2026-04-07T01:12:27.963702+00:…","""51W1AoWLo2U"""
1,1,"""ChwKGkNPZkl0dUQ5dnBNREZUUFFsQW…","""0:03""",3.053,"""60ab99ad""","""Bora veeer""",null,"""text""","""2026-03-26T23:00:04.408408""","""2026-04-07T01:12:27.963702+00:…","""51W1AoWLo2U"""
2,2,"""ChwKGkNOT0NxT0g5dnBNREZhX0JsQW…","""0:04""",4.998,"""6ba1fa45""","""Angelo fulgini vai destruir a …",null,"""text""","""2026-03-26T23:00:06.273171""","""2026-04-07T01:12:27.963702+00:…","""51W1AoWLo2U"""
3,3,"""ChwKGkNNemtqdWI5dnBNREZhekNsQW…","""0:14""",14.948,"""e57e3945""","""jamaica vai fumar os jogadores…",null,"""text""","""2026-03-26T23:00:16.345364""","""2026-04-07T01:12:27.963702+00:…","""51W1AoWLo2U"""
4,4,"""ChwKGkNQNnBqT2o5dnBNREZWSUJoQV…","""0:19""",19.196,"""de909186""","""Caledônia vai tomar taca""",null,"""text""","""2026-03-26T23:00:20.501012""","""2026-04-07T01:12:27.963702+00:…","""51W1AoWLo2U"""


In [23]:
# Phase 4 — Cell 4.4: Temporal stratified GT sampling (uniform across time slices)

gt_human_frames:     list[pl.DataFrame] = []
gt_consensus_frames: list[pl.DataFrame] = []

n_human_per_seg = max(1, gt_per_video // n_time_segments)
n_cons_per_seg  = max(1, (gt_per_video // 2) // n_time_segments)

print(f"{'video_id':<16} {'segs':>4}  {'human':>6}  {'consensus':>10}  time range")
print("-" * 70)

for v in videos:
    vid = cleaned.filter(pl.col("video_id") == v["video_id"])
    t_min = float(vid["time_in_seconds"].min())
    t_max = float(vid["time_in_seconds"].max())
    slice_width = (t_max - t_min) / n_time_segments

    human_slices:     list[pl.DataFrame] = []
    consensus_slices: list[pl.DataFrame] = []

    for seg in range(n_time_segments):
        seg_start = t_min + seg * slice_width
        seg_end   = seg_start + slice_width if seg < n_time_segments - 1 else t_max + 1
        segment   = vid.filter(
            (pl.col("time_in_seconds") >= seg_start) &
            (pl.col("time_in_seconds") <  seg_end)
        )
        take_h = min(n_human_per_seg, segment.shape[0])
        take_c = min(n_cons_per_seg,  max(0, segment.shape[0] - take_h))
        if take_h + take_c == 0:
            continue
        sampled = segment.sample(n=take_h + take_c, seed=random_seed + seg, shuffle=True)
        human_slices.append(sampled.head(take_h))
        if take_c > 0:
            consensus_slices.append(sampled.tail(take_c))

    v_human = pl.concat(human_slices) if human_slices else pl.DataFrame()
    v_cons  = pl.concat(consensus_slices) if consensus_slices else pl.DataFrame()
    gt_human_frames.append(v_human)
    gt_consensus_frames.append(v_cons)

    t_range = f"{t_min/3600:.2f}h – {t_max/3600:.2f}h"
    print(f"{v['video_id']:<16} {n_time_segments:>4}  {v_human.shape[0]:>6}  {v_cons.shape[0]:>10}  {t_range}")

gt_human_set    = pl.concat([f for f in gt_human_frames    if f.shape[0] > 0])
gt_consensus_set = pl.concat([f for f in gt_consensus_frames if f.shape[0] > 0])
gt_pool         = pl.concat([gt_human_set, gt_consensus_set])

print("-" * 70)
print(f"GT pool: {gt_pool.shape[0]}  |  Human: {gt_human_set.shape[0]}  |  Consensus: {gt_consensus_set.shape[0]}")


video_id         segs   human   consensus  time range
----------------------------------------------------------------------
51W1AoWLo2U         5      50          25  0.00h – 2.98h
KLzlxfxKzec         5      50          25  0.02h – 1.54h
OO-Qhz8lN44         5      50          25  0.02h – 1.47h
ewYXmeND0Zs         5      50          25  0.02h – 2.13h
fkGZsNYTugU         5      50          25  0.00h – 1.36h
g_Tfodf9bmk         5      50          25  0.00h – 4.48h
wYzzWC8L33Q         5      50          25  0.00h – 2.09h
yfyhoO9qse4         5      50          25  0.02h – 2.82h
----------------------------------------------------------------------
GT pool: 600  |  Human: 400  |  Consensus: 200


In [24]:
# Phase 4 — Cell 4.5: Export GT sample to Drive for local annotation
#
# Download gt_sample.csv to your local machine, then run:
#   uv run annotate --input ./output/gt_sample.csv --output ./output/ground_truth_human.csv \
#                   --context ./output/<video_id>/chat_messages.csv
# Upload ground_truth_human.csv back to Drive after annotation.

gt_sample_path = GT_DIR / "gt_sample.csv"
gt_human_set.select(["row_id", "video_id", "message", "timestamp", "time_in_seconds"]).write_csv(str(gt_sample_path))
print(f"Exported {gt_human_set.shape[0]} messages for annotation -> {gt_sample_path}")

gt_consensus_pool_path = GT_DIR / "consensus_pool.csv"
gt_consensus_set.select(["row_id", "video_id", "message", "timestamp", "time_in_seconds"]).write_csv(str(gt_consensus_pool_path))
print(f"Exported {gt_consensus_set.shape[0]} messages for consensus proxy -> {gt_consensus_pool_path}")

gt_split_meta = {
    "sampling": {
        "method": "temporal_stratified_uniform",
        "n_videos": len(videos),
        "gt_per_video": gt_per_video,
        "n_time_segments": n_time_segments,
    },
    "gt_pool": {
        "total_size": gt_pool.shape[0],
        "human_size": gt_human_set.shape[0],
        "consensus_size": gt_consensus_set.shape[0],
    },
    "videos": [
        {
            "video_id": v["video_id"],
            "title": v["metadata"].get("title", ""),
            "human_count": gt_human_set.filter(pl.col("video_id") == v["video_id"]).shape[0],
            "consensus_count": gt_consensus_set.filter(pl.col("video_id") == v["video_id"]).shape[0],
        }
        for v in videos
    ],
}
Path(str(gt_sample_path) + ".meta.json").write_text(json.dumps(gt_split_meta, indent=2))
print("Metadata sidecar written")


Exported 400 messages for annotation -> /data/ground_truth/gt_sample.csv
Exported 200 messages for consensus proxy -> /data/ground_truth/consensus_pool.csv
Metadata sidecar written


In [25]:
# Phase 4 — Cell 4.6: Export cleaned data + load Human GT from Drive

cleaned_path = CLEANED_DIR / "cleaned_data.parquet"

if cleaned_path.exists():
    # Fresh runtime — load from Drive instead of re-exporting
    cleaned = pl.read_parquet(str(cleaned_path))
    print(f"Loaded cleaned data from Drive: {cleaned.shape[0]:,} rows")
else:
    # First-time run — export and write metadata
    cleaned.write_parquet(str(cleaned_path), compression="zstd")
    print(f"Exported cleaned data: {cleaned.shape[0]:,} rows -> {cleaned_path}")
    survival_rate = rows_after / rows_before * 100
    cleaned_meta = {
        "rows_before": rows_before,
        "rows_after": rows_after,
        "survival_rate_pct": round(survival_rate, 2),
        "text_length_mean": round(float(text_lengths.mean()), 1),
        "text_length_median": float(text_lengths.median()),
        "video_id": "multi_video",
        "video_title": "multi_video_dataset",
    }
    Path(str(cleaned_path) + ".meta.json").write_text(json.dumps(cleaned_meta, indent=2))

human_gt_path = GT_DIR / "ground_truth_human.csv"
assert human_gt_path.exists(), (
    f"ground_truth_human.csv not found at {human_gt_path}. "
    "Download gt_sample.csv, annotate locally with `uv run annotate`, and upload back to Drive."
)
gt_human = pl.read_csv(str(human_gt_path))
print(f"\nLoaded Human GT: {gt_human.shape[0]} rows")
print(gt_human.head(5))

Exported cleaned data: 28,533 rows -> /data/cleaned_data/cleaned_data.parquet

Loaded Human GT: 400 rows
shape: (5, 3)
┌────────┬──────────┬─────────────┐
│ row_id ┆ label    ┆ aspect      │
│ ---    ┆ ---      ┆ ---         │
│ i64    ┆ str      ┆ str         │
╞════════╪══════════╪═════════════╡
│ 27056  ┆ Neutral  ┆ unknown     │
│ 8828   ┆ Negative ┆ content     │
│ 14243  ┆ Neutral  ┆ audio_video │
│ 8838   ┆ Neutral  ┆ unknown     │
│ 27862  ┆ Positive ┆ content     │
└────────┴──────────┴─────────────┘


In [26]:
# Phase 4 — Cell 4.7: Human GT verification

VALID_LABELS = ("Positive", "Negative", "Neutral")
VALID_ASPECTS = ("content", "presenter", "community", "audio_video", "meta", "unknown")

assert gt_human.shape[0] == gt_human_size, f"Expected {gt_human_size} rows, got {gt_human.shape[0]}"
assert set(gt_human["label"].unique().to_list()).issubset(set(VALID_LABELS)), (
    f"Invalid labels: {set(gt_human['label'].unique().to_list()) - set(VALID_LABELS)}"
)
assert set(gt_human["aspect"].unique().to_list()).issubset(set(VALID_ASPECTS)), (
    f"Invalid aspects: {set(gt_human['aspect'].unique().to_list()) - set(VALID_ASPECTS)}"
)
assert gt_human.null_count().sum_horizontal().to_list()[0] == 0, "Null values found in Human GT"

label_counts = gt_human.group_by("label").agg(pl.len().alias("count")).sort("count", descending=True)
print("Sentiment distribution:")
for row in label_counts.iter_rows(named=True):
    flag = " \u26a0 LOW" if row["count"] < 30 else ""
    print(f"  {row['label']}: {row['count']}{flag}")

aspect_counts = gt_human.group_by("aspect").agg(pl.len().alias("count")).sort("count", descending=True)
print("\nAspect distribution:")
for row in aspect_counts.iter_rows(named=True):
    print(f"  {row['aspect']}: {row['count']}")

print("\n\u2705 Human GT validation passed")

# Per-video label distribution
if "video_id" in gt_human.columns:
    print("\nPer-video label counts:")
    print(gt_human.group_by(["video_id", "label"]).len().sort(["video_id", "label"]))

Sentiment distribution:
  Neutral: 244
  Negative: 80
  Positive: 76

Aspect distribution:
  content: 243
  unknown: 94
  community: 34
  presenter: 17
  meta: 6
  audio_video: 6

✅ Human GT validation passed


---
## Phase 5: LLM Classification Pipeline

Comparing 3 local open-source models via vLLM 0.18.1 with `StructuredOutputsParams` for guaranteed JSON output.

**Prompting strategies:** Zero-shot, ICL (6 examples), Chain-of-Thought

**Total:** 3 models × 3 scenarios × 5 runs = 45 classification runs. Zero API cost.

In [27]:
# Phase 5 — Cell 5.1: Configuration

MODELS: dict[str, str] = {
    # ── ≤4B (T4-safe at float16, ~4-8 GiB weights) ──────────────────────────
    "qwen3.5-4b":       "Qwen/Qwen3.5-4B",
    "phi4-mini":        "microsoft/Phi-4-mini-instruct",
    "qwen2.5-3b":       "Qwen/Qwen2.5-3B-Instruct",
    "llama3.2-3b":      "meta-llama/Llama-3.2-3B-Instruct",
    "deepseek-r1-1.5b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "gemma2-2b":        "google/gemma-2-2b-it",
    # ── 7B-9B (T4 marginal — ~14-18 GiB weights; use quantized or local only) ─
    "deepseek-r1-7b":   "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "qwen2.5-7b":       "Qwen/Qwen2.5-7B-Instruct",
    "llama3.1-8b":      "meta-llama/Llama-3.1-8B-Instruct",
    "aya-expanse-8b":   "CohereForAI/aya-expanse-8b",
    "gemma2-9b":        "google/gemma-2-9b-it",
}
# CONVENTION: model_id in Parquets stores the dict KEY (slug), not the VALUE (HF ID).

# Ollama pull tags — used when USE_VLLM=False (local Docker mode)
OLLAMA_MODELS: dict[str, str] = {
    "qwen3.5-4b":       "qwen2.5:3b",         # qwen2.5:4b does not exist (skips 3b→7b); 3b is closest (1.9 GB)
    "phi4-mini":        "phi4-mini:latest",    # Microsoft Phi-4 Mini (2.5 GB)
    "deepseek-r1-7b":   "deepseek-r1:8b",     # 8b already installed (5.2 GB)
    "qwen2.5-3b":       "qwen2.5:3b",         # Qwen 2.5 3B Instruct (1.9 GB)
    "llama3.2-3b":      "llama3.2:3b",        # Meta Llama 3.2 3B Instruct (~2.0 GB)
    "deepseek-r1-1.5b": "deepseek-r1:1.5b",   # DeepSeek-R1-Distill-Qwen-1.5B (~1.1 GB)
    "gemma2-2b":        "gemma2:2b",           # Google Gemma 2 2B Instruct (~1.6 GB)
    "qwen2.5-7b":       "qwen2.5:7b",          # Qwen 2.5 7B Instruct (~4.7 GB Q4)
    "llama3.1-8b":      "llama3.1:8b",         # Meta Llama 3.1 8B Instruct (~4.9 GB Q4, already installed)
    "aya-expanse-8b":   "aya-expanse:8b",       # Cohere Aya Expanse 8B (~4.9 GB Q4)
    "gemma2-9b":        "gemma2:9b",            # Google Gemma 2 9B Instruct (~5.4 GB Q4)
}
# e.g., "qwen3.5-4b" — NOT "Qwen/Qwen3.5-4B". Phase 6 groupby relies on this.

SCENARIOS: list[str] = ["zero_shot", "icl", "cot"]
NUM_RUNS = 5  # @param {type:"slider", min:1, max:10, step:1}
TEMPERATURE = 0.3  # @param {type:"slider", min:0.0, max:1.0, step:0.1}
# temperature > 0 required for meaningful variance analysis across 5 runs.
# 0.3 is low enough for consistency, high enough for measurable stochasticity.

ICL_HOLDOUT_SIZE = 18
ICL_EXAMPLES_PER_CLASS = 2
TOTAL_CELLS = len(MODELS) * len(SCENARIOS) * NUM_RUNS
print(f"Classification matrix: {len(MODELS)} models x {len(SCENARIOS)} scenarios x {NUM_RUNS} runs = {TOTAL_CELLS} cells")

Classification matrix: 3 models x 3 scenarios x 5 runs = 45 cells


In [28]:
# Phase 5 — Cell 5.2: Pydantic schemas, prompt strategies, VRAM management

import gc
import time
import random
import json as _json
import requests as _requests
from dataclasses import dataclass, field
from typing import Literal, Callable
from pydantic import BaseModel, Field, ConfigDict
from pydantic import ValidationError

if USE_VLLM:
    import torch
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import StructuredOutputsParams


class LLMOutput(BaseModel):
    """Schema for vLLM constrained decoding — only fields the LLM generates."""
    model_config = ConfigDict(strict=True)
    label: Literal["Positive", "Negative", "Neutral"]
    aspect: Literal["content", "presenter", "community", "audio_video", "meta", "unknown"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str = Field(min_length=10)


class SentimentResult(BaseModel):
    """Full classification result with metadata injected after inference."""
    model_config = ConfigDict(strict=True)
    label: Literal["Positive", "Negative", "Neutral"]
    aspect: Literal["content", "presenter", "community", "audio_video", "meta", "unknown"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str = Field(min_length=10)
    row_id: int = Field(ge=0)
    model_id: str = Field(default="")
    scenario: Literal["zero_shot", "icl", "cot"] = Field(default="zero_shot")
    run_id: int = Field(default=1, ge=1, le=10)
    latency_ms: float = Field(default=0.0, ge=0.0)


def clear_vram() -> None:
    """Release GPU memory between model loads (no-op in local/CPU mode).

    In InprocClient mode (VLLM_ENABLE_V1_MULTIPROCESSING=0), torch.distributed
    runs in the main process and is NOT torn down by del llm. Explicitly
    destroy the process group and reset vLLM parallel state so the next model
    load sees a clean CUDA context.

    @returns: None. Prints warning if >2 GiB still allocated after clearing.
    Time: O(1). Space: O(1).
    """
    gc.collect()
    if not USE_VLLM:
        return
    # Destroy distributed process group if alive (InprocClient leaves it open)
    try:
        import torch.distributed as _dist
        if _dist.is_initialized():
            _dist.destroy_process_group()
    except Exception:
        pass
    # Reset vLLM cached parallel state so next LLM() initialises cleanly
    try:
        import vllm.distributed.parallel_state as _vps
        _vps._WORLD = None
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
        if allocated_gb >= 2.0:
            print(f"[WARN] {allocated_gb:.2f} GiB still allocated after cache clear")


def ollama_classify(prompt: str, ollama_tag: str) -> str:
    """Classify one message via Ollama REST API with JSON schema enforcement.

    @param prompt: Formatted prompt string.
    @param ollama_tag: Ollama model tag (e.g. 'qwen2.5:4b').
    @returns: Raw JSON string from model output.
    @throws: requests.RequestException on network failure.
    Time: O(output_tokens). Space: O(output_tokens).
    """
    resp = _requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": ollama_tag,
            "prompt": prompt,
            "format": LLMOutput.model_json_schema(),
            "stream": False,
            "options": {"temperature": TEMPERATURE, "num_predict": 512},
        },
        timeout=180,
    )
    resp.raise_for_status()
    return resp.json()["response"]


def safe_parse_output(
    raw_text: str, row_id: int, model_slug: str, scenario: str, run_id: int, latency_ms: float,
) -> SentimentResult | None:
    """Parse vLLM output into SentimentResult with graceful error handling.

    @param raw_text: Raw JSON from vLLM.
    @param row_id: Source row identifier.
    @param model_slug: Short model slug (e.g., 'qwen3.5-4b').
    @param scenario: Classification scenario.
    @param run_id: Repetition index.
    @param latency_ms: Classification latency.
    @returns: SentimentResult on success, None on failure.
    Time: O(1). Space: O(R).
    """
    try:
        data = _json.loads(raw_text)
        # Clamp confidence to valid range — vLLM constrained decoding enforces JSON
        # structure but not number ranges; models may generate values outside [0, 1].
        if "confidence" in data:
            data["confidence"] = max(0.0, min(1.0, float(data["confidence"])))
        # Pad reasoning to meet min_length=10 — models with no reasoning examples
        # (e.g. ICL scenario) may generate very short strings.
        reasoning = str(data.get("reasoning", ""))
        if len(reasoning) < 10:
            data["reasoning"] = (reasoning + " [no reasoning]")[:max(10, len(reasoning) + 14)]
        llm_output = LLMOutput.model_validate(data)
        return SentimentResult(
            label=llm_output.label,
            aspect=llm_output.aspect,
            confidence=llm_output.confidence,
            reasoning=llm_output.reasoning,
            row_id=row_id,
            model_id=model_slug,
            scenario=scenario,
            run_id=run_id,
            latency_ms=latency_ms,
        )
    except (ValidationError, ValueError, KeyError) as parse_error:
        err_count = parse_error.error_count() if isinstance(parse_error, ValidationError) else 1
        print(f"[SKIP] Row {row_id}: parse failed — {err_count} error(s): {parse_error}")
        return None


# --- Prompt Strategy Functions ---

SYSTEM_PROMPT_ZERO_SHOT: str = (
    "You are a sentiment and aspect classifier for Portuguese-language YouTube live stream chat messages. "
    "These messages are short, informal, and may contain Portuguese slang, emojis, sarcasm, "
    "internet abbreviations (kkkk = laughter, rs/rsrs = laughter, pqp = frustration), "
    "and football/soccer terminology. "
    "Classify each message on two dimensions:\n"
    "1. Sentiment: Positive, Negative, or Neutral\n"
    "2. Aspect (what the message refers to): content, presenter, community, audio_video, or meta\n\n"
    "Aspect definitions:\n"
    "- content: about the stream topic/event (goals, plays, match events)\n"
    "- presenter: about the host/streamer (commentary quality, personality)\n"
    "- community: about other viewers/chat (chat behavior, other users)\n"
    "- audio_video: about stream quality (lag, sound, image)\n"
    "- meta: off-topic, greetings, self-referential, or uncategorizable\n""- unknown: cannot determine the aspect from the message — too short, ambiguous, or decontextualized\n\n"
    "IMPORTANT: Brazilian profanity (caralho, porra, pqp) is often used to express POSITIVE excitement "
    "in sports context, not negativity. Classify by true intent, not surface words. "
    "Provide a confidence score between 0 and 1, and explain your reasoning."
)

SYSTEM_PROMPT_ICL: str = (
    "You are a sentiment and aspect classifier for Portuguese-language YouTube live stream chat messages. "
    "Classify each message on two dimensions:\n"
    "1. Sentiment: Positive, Negative, or Neutral\n"
    "2. Aspect: content, presenter, community, audio_video, meta, or unknown\n\n"
    "You will be given labeled examples first, then a new message to classify. "
    "Follow the same classification approach as the examples."
)

SYSTEM_PROMPT_COT: str = (
    "You are a sentiment and aspect classifier for Portuguese-language YouTube live stream chat messages. "
    "These messages are short, informal, and may contain Portuguese slang, emojis, sarcasm, "
    "internet abbreviations (kkkk = laughter, rs/rsrs = laughter, pqp = frustration), "
    "and football/soccer terminology. "
    "Classify each message on two dimensions:\n"
    "1. Sentiment: Positive, Negative, or Neutral\n"
    "2. Aspect: content, presenter, community, audio_video, meta, or unknown\n\n"
    "Before deciding, follow these reasoning steps:\n"
    "Step 1: Identify the key sentiment-bearing words and phrases. Note any sarcasm, irony, or negation.\n"
    "Step 2: Determine what the message is ABOUT — the stream event (content), the host (presenter), "
    "other viewers (community), technical quality (audio_video), or something off-topic (meta).\n"
    "Step 3: Weigh the overall sentiment signal and assign a label with confidence.\n\n"
    "Write your full step-by-step reasoning in the reasoning field."
)

PromptBuilder = Callable[[str, int], str]


def build_zero_shot_prompt(text: str, row_id: int) -> str:
    """Build a zero-shot classification prompt.

    @param text: Chat message to classify.
    @param row_id: Source row identifier.
    @returns: Formatted prompt string.
    Time: O(T). Space: O(T + S).
    """
    if not text or not text.strip():
        raise ValueError(f"Row {row_id}: Input text must not be empty")
    return f"<|system|>{SYSTEM_PROMPT_ZERO_SHOT}<|end|><|user|>{text}<|end|><|assistant|>"


def build_cot_prompt(text: str, row_id: int) -> str:
    """Build a chain-of-thought classification prompt.

    @param text: Chat message to classify.
    @param row_id: Source row identifier.
    @returns: Formatted prompt string.
    Time: O(T). Space: O(T + S).
    """
    if not text or not text.strip():
        raise ValueError(f"Row {row_id}: Input text must not be empty")
    return f"<|system|>{SYSTEM_PROMPT_COT}<|end|><|user|>{text}<|end|><|assistant|>"


def select_icl_examples(
    holdout_pool: list[dict[str, str]], examples_per_class: int = 2, seed: int | None = None,
) -> list[dict[str, str]]:
    """Select stratified ICL examples from the holdout pool.

    @param holdout_pool: 18-message pool with 'text', 'label', 'aspect' keys.
    @param examples_per_class: Examples per sentiment class.
    @param seed: Random seed (use run_id).
    @returns: Shuffled list of selected examples.
    Time: O(P). Space: O(C).
    """
    rng = random.Random(seed)
    label_groups: dict[str, list[dict[str, str]]] = {}
    for example in holdout_pool:
        label_groups.setdefault(example["label"], []).append(example)
    selected: list[dict[str, str]] = []
    for label_name in sorted(label_groups.keys()):
        selected.extend(rng.sample(label_groups[label_name], examples_per_class))
    rng.shuffle(selected)
    return selected


def build_icl_prompt(text: str, row_id: int, examples: list[dict[str, str]]) -> str:
    """Build an ICL prompt with in-context examples.

    @param text: Chat message to classify.
    @param row_id: Source row identifier.
    @param examples: Pre-selected ICL examples.
    @returns: Formatted prompt string.
    Time: O(E + T). Space: O(E + T + S).
    """
    if not text or not text.strip():
        raise ValueError(f"Row {row_id}: Input text must not be empty")
    lines = ["Here are some labeled examples:", ""]
    for i, ex in enumerate(examples, 1):
        lines.extend([
            f"Example {i}:",
            f"Text: {ex['text']}",
            f"Sentiment: {ex['label']}",
            f"Aspect: {ex['aspect']}",
            "",
        ])
    lines.append(
        "Now classify the following text. Respond ONLY with valid JSON "
        "{label, aspect, confidence (0.0-1.0), reasoning (explain in 1-2 sentences)}:"
    )
    examples_section = "\n".join(lines)
    return f"<|system|>{SYSTEM_PROMPT_ICL}<|end|><|user|>{examples_section}\n\nText: {text}<|end|><|assistant|>"


def create_icl_prompt_builder(holdout_pool: list[dict[str, str]], run_id: int) -> PromptBuilder:
    """Create a bound ICL prompt builder for a specific run.

    @param holdout_pool: 18-message holdout pool.
    @param run_id: Repetition index (used as seed).
    @returns: PromptBuilder callable.
    Time: O(P). Space: O(E).
    """
    examples = select_icl_examples(holdout_pool, examples_per_class=ICL_EXAMPLES_PER_CLASS, seed=run_id)
    def icl_builder(text: str, row_id: int) -> str:
        return build_icl_prompt(text, row_id, examples)
    return icl_builder


STRATEGY_REGISTRY: dict[str, PromptBuilder] = {
    "zero_shot": build_zero_shot_prompt,
    "cot": build_cot_prompt,
}

print("Pydantic schemas, prompt strategies, and VRAM management defined")


Pydantic schemas, prompt strategies, and VRAM management defined


In [29]:
# Phase 5 — Cell 5.3: ICL holdout selection + evaluation set preparation

# Load consensus pool from Drive on fresh runtimes (defined by Cell 4.4 on first run)
if 'gt_consensus_set' not in dir():
    _consensus_path = GT_DIR / 'consensus_pool.csv'
    assert _consensus_path.exists(), (
        f'consensus_pool.csv not found at {_consensus_path}. Run Cell 4.4 first.'
    )
    gt_consensus_set = pl.read_csv(str(_consensus_path))
    print(f'Loaded consensus pool from Drive: {gt_consensus_set.shape[0]} rows')

gt_human_with_text = gt_human.join(
    cleaned.select(["row_id", "message"]),
    on="row_id",
    how="inner",
)

holdout_messages: list[dict[str, str]] = []
for label_value in sorted(VALID_LABELS):
    label_subset = gt_human_with_text.filter(pl.col("label") == label_value)
    sampled = label_subset.sample(n=min(6, label_subset.shape[0]), seed=random_seed)
    for row in sampled.iter_rows(named=True):
        holdout_messages.append({
            "row_id": str(row["row_id"]),
            "text": row["message"],
            "label": row["label"],
            "aspect": row["aspect"],
        })

holdout_row_ids = [int(m["row_id"]) for m in holdout_messages]
holdout_path = GT_DIR / "icl_holdout_row_ids.json"
holdout_path.write_text(json.dumps(sorted(holdout_row_ids)))

eval_gt = gt_human.filter(~pl.col("row_id").is_in(holdout_row_ids))
eval_row_ids = eval_gt["row_id"].to_list()
eval_texts_df = cleaned.filter(pl.col("row_id").is_in(eval_row_ids)).sort("row_id")
eval_texts = eval_texts_df["message"].to_list()
eval_row_ids_ordered = eval_texts_df["row_id"].to_list()

consensus_texts_df = cleaned.filter(
    pl.col("row_id").is_in(gt_consensus_set["row_id"].to_list())
).sort("row_id")
consensus_texts = consensus_texts_df["message"].to_list()
consensus_row_ids = consensus_texts_df["row_id"].to_list()

all_classify_texts = eval_texts + consensus_texts
all_classify_row_ids = eval_row_ids_ordered + consensus_row_ids

print(f"ICL holdout: {len(holdout_messages)} messages ({', '.join(str(r) for r in sorted(holdout_row_ids)[:5])}...)")
print(f"Holdout by class: { {m['label']: sum(1 for x in holdout_messages if x['label'] == m['label']) for m in holdout_messages} }")
print(f"Evaluation set: {len(eval_texts)} messages ({gt_human_size} - {len(holdout_row_ids)} holdout)")
print(f"Consensus pool: {len(consensus_texts)} messages")
print(f"Total to classify per run: {len(all_classify_texts)} messages")

for label_value in VALID_LABELS:
    count = eval_gt.filter(pl.col("label") == label_value).shape[0]
    assert count >= 20, f"Class '{label_value}' has only {count} eval examples after holdout (need >=20)"
    print(f"  Eval class '{label_value}': {count} examples")

ICL holdout: 18 messages (8929, 9038, 9138, 14297, 14749...)
Holdout by class: {'Negative': 6, 'Neutral': 6, 'Positive': 6}
Evaluation set: 382 messages (400 - 18 holdout)
Consensus pool: 200 messages
Total to classify per run: 582 messages
  Eval class 'Positive': 70 examples
  Eval class 'Negative': 74 examples
  Eval class 'Neutral': 238 examples


In [ ]:
%%time
# Phase 5 — Cell 5.4: Run 3x3x5 classification matrix with checkpoint/resume
#
# Iterates MODELS x SCENARIOS x NUM_RUNS. Each (model, scenario, run) cell is
# classified in one batch (vLLM) or sequentially (Ollama). Checkpoints atomically
# after every cell; resumes from existing parquet on re-run. KeyboardInterrupt
# saves partial progress before propagating.
# Time: O(M * S * R * N) where M=models, S=scenarios, R=runs, N=messages.
# Space: O(M * S * R * N) for all_results accumulation.

import threading
import subprocess as _sp
from tqdm.auto import tqdm


def _vram_heartbeat(stop_evt: threading.Event, label: str) -> None:
    """Poll nvidia-smi and print VRAM usage until stopped.

    @param stop_evt: Threading event; set to stop the heartbeat loop.
    @param label: Display label for log lines (e.g. model slug).
    @returns: None (side-effect only: prints to stdout every 12 s).
    Time: O(K) where K = number of heartbeat ticks. Space: O(1).
    """
    t0 = time.monotonic()
    while not stop_evt.wait(timeout=12):
        elapsed = int(time.monotonic() - t0)
        r = _sp.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            used, total = [int(x.strip()) for x in r.stdout.strip().split(',')]
            print(f'  [{elapsed:3d}s] {label} \u2014 VRAM {used/1024:.1f}/{total/1024:.1f} GiB', flush=True)


def _load_llm(model_hf_id: str, label: str, **kwargs) -> 'LLM':
    """Instantiate vLLM LLM with a VRAM heartbeat running during weight loading.

    @param model_hf_id: HuggingFace model identifier.
    @param label: Display label for progress banners.
    @param kwargs: Forwarded to vllm.LLM constructor.
    @returns: Initialised LLM instance.
    @throws torch.cuda.OutOfMemoryError: If GPU memory is insufficient.
    Time: O(W) where W = model weight loading time. Space: O(W).
    """
    print(f"\n{'='*64}")
    print(f'[LOAD] {label}')
    print(f'       model : {model_hf_id}')
    print(f'       VRAM updates every 12 s until weights are loaded')
    print(f"{'='*64}", flush=True)
    stop_evt = threading.Event()
    hb = threading.Thread(target=_vram_heartbeat, args=(stop_evt, label), daemon=True)
    hb.start()
    t0 = time.monotonic()
    # vLLM's suppress_stdout() calls sys.stdout.fileno() which raises
    # UnsupportedOperation in Jupyter (OutStream has no real fd).
    # Patch the reference in parallel_state's module namespace — the only
    # place that calls suppress_stdout in InprocClient (VLLM_ENABLE_V1_MULTIPROCESSING=0).
    import contextlib as _cl, os as _os, sys as _sys
    import vllm.distributed.parallel_state as _vps
    if not getattr(_vps, "_suppress_stdout_patched", False):
        @_cl.contextmanager
        def _jupyter_safe_suppress_stdout():
            try:
                fd = _sys.stdout.fileno()
                stdout_dup = _os.dup(fd)
                devnull_fd = _os.open(_os.devnull, _os.O_WRONLY)
                try:
                    _os.dup2(devnull_fd, fd)
                    yield
                finally:
                    _os.dup2(stdout_dup, fd)
                    _os.close(stdout_dup)
                    _os.close(devnull_fd)
            except Exception:
                yield  # Jupyter stdout: no real fd — skip suppression
        _vps.suppress_stdout = _jupyter_safe_suppress_stdout
        _vps._suppress_stdout_patched = True
    try:
        result = LLM(model=model_hf_id, **kwargs)
    finally:
        stop_evt.set()
        hb.join(timeout=5.0)
    elapsed = time.monotonic() - t0
    r = _sp.run(
        ['nvidia-smi', '--query-gpu=memory.used,memory.total',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True,
    )
    vram_info = ''
    if r.returncode == 0:
        used, total = [int(x.strip()) for x in r.stdout.strip().split(',')]
        vram_info = f' \u2014 VRAM {used/1024:.1f}/{total/1024:.1f} GiB'
    print(f'[LOAD] {label} ready in {elapsed:.1f}s{vram_info}', flush=True)
    return result


def _save_checkpoint(results: list[dict], path) -> None:
    """Atomically write classification results to a parquet checkpoint.

    @param results: List of result dicts to persist.
    @param path: Target parquet file path.
    @returns: None. Writes to a temp file then renames (atomic on same fs).
    Time: O(N) where N = len(results). Space: O(N).
    """
    from pathlib import Path
    path = Path(path)
    tmp = path.with_suffix('.parquet.tmp')
    pl.DataFrame(results).write_parquet(str(tmp))
    tmp.rename(path)


all_results: list[dict] = []
progress = tqdm(total=TOTAL_CELLS, desc='Classification matrix', unit='cell')

for model_slug, model_hf_id in MODELS.items():
    model_dir = CLASSIFIER_DIR / model_slug
    model_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = model_dir / 'checkpoint.parquet'
    completed_cells: set[tuple[str, int]] = set()
    model_results: list[dict] = []

    if checkpoint_path.exists():
        existing = pl.read_parquet(str(checkpoint_path))
        if existing.height > 0:
            model_results = existing.to_dicts()
            completed_cells = set(
                existing.select('scenario', 'run_id').unique().iter_rows()
            )
            print(f'[RESUME] {model_slug}: {len(completed_cells)}/15 cells already complete')
        else:
            print(f'[RESUME] {model_slug}: empty checkpoint, starting fresh')

    if len(completed_cells) == len(SCENARIOS) * NUM_RUNS:
        print(f'[SKIP] {model_slug} fully complete')
        all_results.extend(model_results)
        progress.update(len(SCENARIOS) * NUM_RUNS)
        continue

    clear_vram()

    if USE_VLLM:
        hf_token = None
        _llm_kwargs = dict(
            dtype=VLLM_DTYPE,
            enforce_eager=ENFORCE_EAGER,
            max_model_len=MAX_MODEL_LEN,
            gpu_memory_utilization=GPU_MEM_UTIL,
            hf_token=hf_token,
        )
        try:
            llm = _load_llm(model_hf_id, model_slug, **_llm_kwargs)
        except OSError as _gated_err:
            _err_str = str(_gated_err)
            if 'gated' in _err_str.lower() or '403' in _err_str:
                print(f'[SKIP] {model_slug}: gated repo — HF access not granted.')
                print(f'       Visit https://huggingface.co/{model_hf_id} to request access.')
                progress.update(len(SCENARIOS) * NUM_RUNS - len(completed_cells))
                continue
            raise
        except (torch.cuda.OutOfMemoryError, RuntimeError) as _load_err:
            print(f'[OOM/ERR] {model_slug}: load failed ({type(_load_err).__name__}), retrying with max_model_len=512...')
            clear_vram()
            llm = _load_llm(
                model_hf_id, f'{model_slug}/retry',
                dtype=VLLM_DTYPE, enforce_eager=True,
                max_model_len=512, gpu_memory_utilization=0.88, hf_token=hf_token,
            )
        structured_params = StructuredOutputsParams(json=LLMOutput.model_json_schema())
        sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=1536, structured_outputs=structured_params)
    else:
        ollama_tag = OLLAMA_MODELS.get(model_slug)
        if not ollama_tag:
            print(f'[SKIP] {model_slug} has no Ollama tag \u2014 skipping')
            continue
        _known = {m['name'] for m in _requests.get(f'{OLLAMA_URL}/api/tags', timeout=10).json().get('models', [])}
        if ollama_tag not in _known:
            print(f'[OLLAMA] Downloading {ollama_tag}...')
            _requests.post(f'{OLLAMA_URL}/api/pull', json={'name': ollama_tag, 'stream': False}, timeout=600)
        else:
            print(f'[OLLAMA] {ollama_tag} already cached, skipping pull')
        llm = None
        sampling = None

    try:
        for scenario in SCENARIOS:
            for run_id in range(1, NUM_RUNS + 1):
                if (scenario, run_id) in completed_cells:
                    progress.update(1)
                    continue

                if scenario == 'icl':
                    prompt_builder = create_icl_prompt_builder(holdout_messages, run_id=run_id)
                else:
                    prompt_builder = STRATEGY_REGISTRY[scenario]

                prompts = [prompt_builder(text, rid) for text, rid in zip(all_classify_texts, all_classify_row_ids)]

                cell_results: list[dict] = []
                per_msg_latency = 0.0
                if USE_VLLM:
                    print(
                        f'[INFER] {model_slug}/{scenario}/run{run_id}: '
                        f'{len(prompts)} prompts \u2192 batch inference...',
                        flush=True,
                    )
                    batch_start = time.perf_counter() * 1000
                    outputs = llm.generate(prompts, sampling)
                    batch_elapsed = time.perf_counter() * 1000 - batch_start
                    per_msg_latency = batch_elapsed / len(prompts)
                    print(
                        f'[INFER] done in {batch_elapsed/1000:.1f}s '
                        f'({per_msg_latency:.0f} ms/msg, {len(outputs)} outputs)',
                        flush=True,
                    )
                    for output, rid in zip(outputs, all_classify_row_ids):
                        parsed = safe_parse_output(
                            raw_text=output.outputs[0].text,
                            row_id=rid,
                            model_slug=model_slug,
                            scenario=scenario,
                            run_id=run_id,
                            latency_ms=per_msg_latency,
                        )
                        if parsed is not None:
                            result_dict = parsed.model_dump()
                            result_dict['predicted_label'] = result_dict.pop('label')
                            result_dict['predicted_aspect'] = result_dict.pop('aspect')
                            cell_results.append(result_dict)
                else:
                    _latencies: list[float] = []
                    msg_iter = tqdm(
                        zip(prompts, all_classify_row_ids),
                        total=len(prompts),
                        desc=f'{model_slug}/{scenario}/run{run_id}',
                        unit='msg',
                        leave=False,
                    )
                    for prompt, rid in msg_iter:
                        t0 = time.perf_counter() * 1000
                        try:
                            raw = ollama_classify(prompt, OLLAMA_MODELS[model_slug])
                        except Exception as e:
                            print(f'[SKIP] Row {rid}: Ollama error \u2014 {e}')
                            continue
                        latency_ms = time.perf_counter() * 1000 - t0
                        _latencies.append(latency_ms)
                        parsed = safe_parse_output(
                            raw_text=raw,
                            row_id=rid,
                            model_slug=model_slug,
                            scenario=scenario,
                            run_id=run_id,
                            latency_ms=latency_ms,
                        )
                        if parsed is not None:
                            result_dict = parsed.model_dump()
                            result_dict['predicted_label'] = result_dict.pop('label')
                            result_dict['predicted_aspect'] = result_dict.pop('aspect')
                            cell_results.append(result_dict)

                    per_msg_latency = sum(_latencies) / max(len(_latencies), 1)

                model_results.extend(cell_results)
                _save_checkpoint(model_results, checkpoint_path)
                progress.update(1)
                progress.set_postfix(
                    model=model_slug, scenario=scenario, run=run_id,
                    classified=len(cell_results), latency=f'{per_msg_latency:.0f}ms',
                )
    except KeyboardInterrupt:
        _save_checkpoint(model_results, checkpoint_path)
        print(f'\n[INTERRUPT] {model_slug}: {len(model_results)} results saved to checkpoint')
        raise

    print(f'[CHECKPOINT] {model_slug}: {len(model_results)} results saved')

    latency_report = {
        'model_id': model_slug,
        'total_classifications': len(model_results),
        'mean_latency_ms': round(sum(r['latency_ms'] for r in model_results) / max(len(model_results), 1), 2),
    }
    (model_dir / f'latency_log_{model_slug}.json').write_text(json.dumps(latency_report, indent=2))

    all_results.extend(model_results)
    if USE_VLLM:
        del llm
    clear_vram()

progress.close()
results_df = pl.DataFrame(all_results)
print(f'\nTotal classifications: {len(results_df):,}')
print(f"Unique (model, scenario, run) combinations: {results_df.select('model_id', 'scenario', 'run_id').unique().shape[0]}")


In [ ]:
# Phase 5 — Cell 5.4b: Targeted retry — re-classify missing rows with max_tokens=1536
#
# For each completed model checkpoint, identifies (model, scenario, run) cells
# that have missing row_ids (due to max_tokens=512 truncation or mid-batch
# Colab disconnects), then re-runs inference ONLY on those rows using
# max_tokens=1536. Merges results atomically into the existing checkpoint.
# Run this after Cell 5.4 is fully complete for all models.
# Time: O(missing_rows * models). Space: O(checkpoint_size).

_RETRY_MAX_TOKENS = 1536
_all_row_ids = set(all_classify_row_ids)
_expected_per_cell = len(all_classify_row_ids)

# ── 1. Diagnostic pass (no GPU needed) ────────────────────────────────────────
print("=== Missing row diagnostics ===")
print(f"Expected rows per (model, scenario, run): {_expected_per_cell}\n")

_models_needing_retry: list[str] = []

for model_slug in MODELS:
    cp = CLASSIFIER_DIR / model_slug / "checkpoint.parquet"
    if not cp.exists():
        print(f"[--] {model_slug}: no checkpoint yet — run Cell 5.4 first")
        continue
    df = pl.read_parquet(str(cp))
    model_total_missing = 0
    for scenario in SCENARIOS:
        for run_id in range(1, NUM_RUNS + 1):
            present = set(
                df.filter(
                    (pl.col("scenario") == scenario) &
                    (pl.col("run_id") == run_id)
                )["row_id"].to_list()
            )
            n_missing = len(_all_row_ids - present)
            model_total_missing += n_missing
            if n_missing > 0:
                print(f"  {model_slug}/{scenario}/run{run_id}: {n_missing} missing  ({_expected_per_cell - n_missing} ok)")
    if model_total_missing > 0:
        print(f"  → {model_slug}: {model_total_missing} rows to retry")
        _models_needing_retry.append(model_slug)
    else:
        print(f"  {model_slug}: complete ✓  (0 missing)")

print(f"\nModels needing retry: {_models_needing_retry or 'none'}")
if not _models_needing_retry:
    print("Nothing to do — all checkpoints are complete.")
    raise SystemExit(0)

# ── 2. Retry loop ──────────────────────────────────────────────────────────────
for model_slug in _models_needing_retry:
    model_hf_id = MODELS[model_slug]
    cp = CLASSIFIER_DIR / model_slug / "checkpoint.parquet"
    existing_df = pl.read_parquet(str(cp))
    existing_results = existing_df.to_dicts()
    new_results: list[dict] = []

    # Build per-row_id lookup for fast prompt retrieval
    _rid_to_text = {rid: txt for rid, txt in zip(all_classify_row_ids, all_classify_texts)}

    print(f"\n{'='*64}")
    print(f"[RETRY] {model_slug}  ({model_hf_id})")
    print(f"{'='*64}")

    clear_vram()
    _llm_kwargs = dict(
        dtype=VLLM_DTYPE,
        enforce_eager=ENFORCE_EAGER,
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=GPU_MEM_UTIL,
    )
    llm = _load_llm(model_hf_id, model_slug, **_llm_kwargs)

    structured_params = StructuredOutputsParams(json=LLMOutput.model_json_schema())
    retry_sampling = SamplingParams(
        temperature=TEMPERATURE,
        max_tokens=_RETRY_MAX_TOKENS,
        structured_outputs=structured_params,
    )

    for scenario in SCENARIOS:
        for run_id in range(1, NUM_RUNS + 1):
            present_ids = set(
                existing_df.filter(
                    (pl.col("scenario") == scenario) &
                    (pl.col("run_id") == run_id)
                )["row_id"].to_list()
            )
            missing_ids = sorted(_all_row_ids - present_ids)
            if not missing_ids:
                print(f"  [OK] {scenario}/run{run_id}: no missing rows")
                continue

            print(f"  [RETRY] {scenario}/run{run_id}: {len(missing_ids)} rows → max_tokens={_RETRY_MAX_TOKENS}", flush=True)

            if scenario == "icl":
                prompt_builder = create_icl_prompt_builder(holdout_messages, run_id=run_id)
            else:
                prompt_builder = STRATEGY_REGISTRY[scenario]

            retry_texts = [_rid_to_text[rid] for rid in missing_ids]
            prompts = [prompt_builder(txt, rid) for txt, rid in zip(retry_texts, missing_ids)]

            batch_start = time.perf_counter() * 1000
            outputs = llm.generate(prompts, retry_sampling)
            batch_elapsed = time.perf_counter() * 1000 - batch_start
            per_msg_latency = batch_elapsed / max(len(prompts), 1)

            cell_new = 0
            for output, rid in zip(outputs, missing_ids):
                parsed = safe_parse_output(
                    raw_text=output.outputs[0].text,
                    row_id=rid,
                    model_slug=model_slug,
                    scenario=scenario,
                    run_id=run_id,
                    latency_ms=per_msg_latency,
                )
                if parsed is not None:
                    d = parsed.model_dump()
                    d["predicted_label"] = d.pop("label")
                    d["predicted_aspect"] = d.pop("aspect")
                    new_results.append(d)
                    cell_new += 1

            print(f"         recovered {cell_new}/{len(missing_ids)} rows  ({len(missing_ids) - cell_new} still unparseable)", flush=True)

    del llm
    clear_vram()

    if new_results:
        merged = existing_results + new_results
        _save_checkpoint(merged, cp)
        print(f"[SAVED] {model_slug}: +{len(new_results)} rows → checkpoint updated ({len(merged)} total)")
    else:
        print(f"[SKIP] {model_slug}: no new parseable results")

# ── 3. Rebuild results_df from all checkpoints ────────────────────────────────
print("\n=== Rebuilding results_df from checkpoints ===")
_all_results: list[dict] = []
for model_slug in MODELS:
    cp = CLASSIFIER_DIR / model_slug / "checkpoint.parquet"
    if cp.exists():
        _all_results.extend(pl.read_parquet(str(cp)).to_dicts())

results_df = pl.DataFrame(_all_results)
print(f"results_df: {len(results_df):,} rows across {results_df['model_id'].n_unique()} models")

# Final coverage report
print("\n=== Coverage after retry ===")
for model_slug in MODELS:
    cp = CLASSIFIER_DIR / model_slug / "checkpoint.parquet"
    if not cp.exists():
        continue
    df = pl.read_parquet(str(cp))
    for scenario in SCENARIOS:
        for run_id in range(1, NUM_RUNS + 1):
            n = df.filter(
                (pl.col("scenario") == scenario) &
                (pl.col("run_id") == run_id)
            ).height
            status = "✓" if n == _expected_per_cell else f"MISSING {_expected_per_cell - n}"
            print(f"  {model_slug}/{scenario}/run{run_id}: {n}/{_expected_per_cell} {status}")


In [ ]:
# Phase 5 — Cell 5.4c: Quality Verifier v3
# ─────────────────────────────────────────────────────────────────────────────
# Run after Cell 5.4b (retry). Auto-discovers all model checkpoints, reports:
#   • Coverage vs expected rows (582) with failure-type classification
#   • Cross-run agreement per scenario (label consistency across 5 runs)
#   • Label distribution with collapse detection
#   • Confidence and latency stats
#   • End-of-cell summary verdict table
# ─────────────────────────────────────────────────────────────────────────────
import polars as pl
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
_CLASSIFIER_DIR    = Path('/content/drive/MyDrive/sentiment_analyzer/classifier_output')
_EXPECTED_PER_CELL = len(all_classify_row_ids) if 'all_classify_row_ids' in dir() else 582
_SCENARIOS         = SCENARIOS if 'SCENARIOS' in dir() else ['zero_shot', 'icl', 'cot']
_NUM_RUNS          = NUM_RUNS  if 'NUM_RUNS'  in dir() else 5

# Coverage failure thresholds
_STRAGGLER_MAX_MISSING = 5    # 1-5 missing  → straggler (noise, acceptable)
_PARTIAL_MAX_MISSING   = 50   # 6-50 missing → partial (retry may help)
# > 50 missing → structural failure (model capability limit, do not retry)

# ── Helpers ───────────────────────────────────────────────────────────────────
def _coverage_flag(n: int, expected: int) -> str:
    missing = expected - n
    if missing == 0:                           return '✓ COMPLETE'
    if missing <= _STRAGGLER_MAX_MISSING:      return f'~ straggler   ({missing} missing)'
    if missing <= _PARTIAL_MAX_MISSING:        return f'⚠ partial     ({missing} missing)'
    return f'✗ STRUCTURAL  ({missing} missing)'

def _cross_run_agreement(df: pl.DataFrame, scenario: str) -> list:
    sub       = df.filter(pl.col('scenario') == scenario)
    available = sorted(sub['run_id'].unique().to_list())
    if len(available) < 2:
        return []
    base  = sub.filter(pl.col('run_id') == available[0]).select(['row_id', 'predicted_label'])
    rows  = []
    for rid in available[1:]:
        other  = sub.filter(pl.col('run_id') == rid).select(['row_id', 'predicted_label'])
        merged = base.join(other, on='row_id', suffix='_r')
        n      = len(merged)
        agreed = merged.filter(pl.col('predicted_label') == pl.col('predicted_label_r')).height
        rows.append((available[0], rid, agreed, agreed / n if n else 0.0))
    return rows

def _agreement_flag(pct: float) -> str:
    if pct >= 0.80: return '✓'
    if pct >= 0.70: return '~'
    if pct >= 0.50: return '⚠'
    return '✗ COLLAPSE'

def _detect_label_collapse(df: pl.DataFrame, scenario: str):
    sub = df.filter(pl.col('scenario') == scenario)
    if sub.is_empty():
        return None
    counts = sub['predicted_label'].value_counts(sort=True)
    top_n  = counts[0, 'count']
    total  = counts['count'].sum()
    if total > 0 and top_n / total > 0.90:
        return counts[0, 'predicted_label']
    return None

# ── Main loop ─────────────────────────────────────────────────────────────────
discovered = sorted([p.parent.name for p in _CLASSIFIER_DIR.glob('*/checkpoint.parquet')])
if not discovered:
    print('No checkpoints found in', _CLASSIFIER_DIR)
    raise SystemExit(0)

print(f'{'='*72}')
print(f'  QUALITY VERIFIER v3  —  {len(discovered)} model(s) found')
print(f'  Expected rows per cell: {_EXPECTED_PER_CELL}')
print(f'{'='*72}')

_verdicts: list[dict] = []

for slug in discovered:
    cp    = _CLASSIFIER_DIR / slug / 'checkpoint.parquet'
    df    = pl.read_parquet(str(cp))
    total = len(df)

    print(f'\n{'-'*72}')
    print(f'  MODEL: {slug}  ({total:,} rows in checkpoint)')
    print(f'{'-'*72}')

    # 1. Coverage per cell
    print('  Coverage per cell:')
    model_ok = model_structural = model_partial = model_straggler = 0
    present_scenarios = df['scenario'].unique().to_list()
    for sc in _SCENARIOS:
        for run in range(1, _NUM_RUNS + 1):
            n = df.filter((pl.col('scenario') == sc) & (pl.col('run_id') == run)).height \
                if sc in present_scenarios else 0
            flag    = _coverage_flag(n, _EXPECTED_PER_CELL)
            missing = _EXPECTED_PER_CELL - n
            print(f'    {sc}/run{run}: {n:>4}/{_EXPECTED_PER_CELL}  {flag}')
            if   missing == 0:                         model_ok         += 1
            elif missing <= _STRAGGLER_MAX_MISSING:    model_straggler  += 1
            elif missing <= _PARTIAL_MAX_MISSING:      model_partial    += 1
            else:                                      model_structural += 1

    # 2. Cross-run agreement
    print('\n  Cross-run agreement (run1 vs others):')
    for sc in _SCENARIOS:
        if sc not in present_scenarios:
            print(f'    {sc}: no data')
            continue
        pairs         = _cross_run_agreement(df, sc)
        collapse_label = _detect_label_collapse(df, sc)
        collapse_note  = f'  ⚠ COLLAPSE→{collapse_label}' if collapse_label else ''
        if not pairs:
            print(f'    {sc}: < 2 runs — skip')
            continue
        for r_a, r_b, agreed, pct in pairs:
            aflag = _agreement_flag(pct)
            note  = collapse_note if r_b == pairs[-1][1] else ''
            print(f'    {sc} run{r_a} vs run{r_b}: {agreed:>4}/{_EXPECTED_PER_CELL} = {pct:.1%}  {aflag}{note}')

    # 3. Label distribution
    print('\n  Label distribution (all scenarios combined):')
    label_dist = (
        df.group_by('predicted_label')
          .agg(pl.len().alias('n'))
          .sort('n', descending=True)
    )
    for row in label_dist.iter_rows(named=True):
        pct = row['n'] / max(total, 1)
        bar = '█' * int(pct * 30)
        print(f"    {row['predicted_label']:<10}  {row['n']:>6,}  {pct:5.1%}  {bar}")

    # 4. Confidence stats
    if 'confidence' in df.columns:
        conf_mean = df['confidence'].mean()
        conf_p10  = df['confidence'].quantile(0.10)
        conf_p90  = df['confidence'].quantile(0.90)
        print(f'\n  Confidence: mean={conf_mean:.3f}  p10={conf_p10:.3f}  p90={conf_p90:.3f}')

    # 5. Latency stats
    if 'latency_ms' in df.columns and df['latency_ms'].mean() is not None:
        lat_mean = df['latency_ms'].mean()
        lat_p95  = df['latency_ms'].quantile(0.95)
        print(f'  Latency:    mean={lat_mean:.0f}ms  p95={lat_p95:.0f}ms')

    _verdicts.append({
        'model':      slug,
        'total_rows': total,
        'complete':   model_ok,
        'straggler':  model_straggler,
        'partial':    model_partial,
        'structural': model_structural,
    })

# ── Summary verdict table ──────────────────────────────────────────────────────
total_cells = _NUM_RUNS * len(_SCENARIOS)
print(f'\n{'='*72}')
print(f'  SUMMARY VERDICT  ({total_cells} cells per model)')
print(f'{'='*72}')
print(f"  {'Model':<22} {'Rows':>7}  {'✓':>3} {'~strag':>6} {'⚠part':>6} {'✗struct':>7}  Verdict")
print(f"  {'-'*22} {'-'*7}  {'-'*3} {'-'*6} {'-'*6} {'-'*7}  {'-'*22}")
for v in _verdicts:
    if v['structural'] > 0:
        verdict = f"NEEDS WORK ({v['structural']} structural cells)"
    elif v['partial'] > 0:
        verdict = f"RETRY ({v['partial']} partial cells)"
    elif v['straggler'] > 0:
        verdict = f"ACCEPTABLE ({v['straggler']} straggler cells)"
    else:
        verdict = 'COMPLETE'
    print(
        f"  {v['model']:<22} {v['total_rows']:>7,}  "
        f"{v['complete']:>3} {v['straggler']:>6} {v['partial']:>6} {v['structural']:>7}  {verdict}"
    )

print(f'\n  Legend:')
print(f'    ✓ COMPLETE   — {_EXPECTED_PER_CELL}/{_EXPECTED_PER_CELL} rows')
print(f'    ~ straggler  — 1–{_STRAGGLER_MAX_MISSING} missing (noise, acceptable)')
print(f'    ⚠ partial    — 6–{_PARTIAL_MAX_MISSING} missing (retry may recover)')
print(f'    ✗ structural — >{_PARTIAL_MAX_MISSING} missing (model capability limit, do not retry)')
print(f'{'='*72}')


In [ ]:
# Phase 5 — Cell 5.4d: Rebuild results_df from checkpoints
#
# Run this on any fresh runtime after Cell 5.3.
# Loads all model checkpoints from Drive into results_df without running inference.

_all: list[dict] = []
for slug in MODELS:
    cp = CLASSIFIER_DIR / slug / "checkpoint.parquet"
    if cp.exists():
        _all.extend(pl.read_parquet(str(cp)).to_dicts())
    else:
        print(f"[WARN] No checkpoint for {slug}")

results_df = pl.DataFrame(_all)
print(f"results_df: {len(results_df):,} rows across {results_df["model_id"].n_unique()} models")
print(f"Cells loaded: {results_df.select("model_id", "scenario", "run_id").unique().shape[0]}/{TOTAL_CELLS}")


In [ ]:
# Phase 5 — Cell 5.5: Export per-run Parquets + combined file

for (model_slug, scenario, run_id), group in results_df.group_by(["model_id", "scenario", "run_id"]):
    model_dir = CLASSIFIER_DIR / model_slug
    model_dir.mkdir(parents=True, exist_ok=True)
    file_path = model_dir / f"classified_{model_slug}_{scenario}_run_{run_id}.parquet"
    group.write_parquet(str(file_path))

run_count = results_df.select("model_id", "scenario", "run_id").unique().shape[0]
print(f"Exported {run_count} individual run Parquets")

results_df.write_parquet(str(CLASSIFIER_DIR / "all_classified.parquet"))
print(f"Combined file: {len(results_df):,} rows -> {CLASSIFIER_DIR / 'all_classified.parquet'}")

# ── Coverage + consistency reports ────────────────────────────────────────────
_STRUCTURAL_THRESHOLD = 50   # >50 missing rows per run → structural failure
_EXPECTED = len(all_classify_row_ids)   # 582

coverage_rows: list[dict] = []
for model_slug in MODELS:
    cp = CLASSIFIER_DIR / model_slug / "checkpoint.parquet"
    if not cp.exists():
        continue
    _df = pl.read_parquet(str(cp))
    for sc in SCENARIOS:
        for run in range(1, NUM_RUNS + 1):
            n       = _df.filter((pl.col("scenario") == sc) & (pl.col("run_id") == run)).height
            missing = _EXPECTED - n
            ftype   = ("COMPLETE"    if missing == 0 else
                       "straggler"   if missing <= 5 else
                       "partial"     if missing <= _STRUCTURAL_THRESHOLD else
                       "STRUCTURAL")
            coverage_rows.append({
                "model": model_slug, "scenario": sc, "run_id": run,
                "n_classified": n, "n_expected": _EXPECTED, "n_missing": missing,
                "coverage_pct": n / _EXPECTED, "failure_type": ftype,
            })

coverage_df = pl.DataFrame(coverage_rows)
coverage_df.write_csv(str(METRICS_DIR / "coverage_report.csv"))

consistency_rows: list[dict] = []
for model_slug in MODELS:
    cp = CLASSIFIER_DIR / model_slug / "checkpoint.parquet"
    if not cp.exists():
        continue
    _df = pl.read_parquet(str(cp))
    for sc in SCENARIOS:
        sub  = _df.filter(pl.col("scenario") == sc)
        runs = sorted(sub["run_id"].unique().to_list())
        if len(runs) < 2:
            continue
        base = sub.filter(pl.col("run_id") == runs[0]).select(["row_id", "predicted_label"])
        for r2 in runs[1:]:
            other  = sub.filter(pl.col("run_id") == r2).select(["row_id", "predicted_label"])
            merged = base.join(other, on="row_id", suffix="_r")
            n      = len(merged)
            agreed = merged.filter(
                pl.col("predicted_label") == pl.col("predicted_label_r")
            ).height
            consistency_rows.append({
                "model": model_slug, "scenario": sc,
                "run_a": runs[0], "run_b": r2,
                "n_agreed": agreed, "n_total": n,
                "agreement_pct": agreed / n if n else 0.0,
            })

consistency_df = pl.DataFrame(consistency_rows)
consistency_df.write_csv(str(METRICS_DIR / "consistency_report.csv"))

print(f"Coverage report:     {len(coverage_df)} cells  → coverage_report.csv")
print(f"Consistency report:  {len(consistency_df)} pairs → consistency_report.csv")
_struct = coverage_df.filter(pl.col("failure_type") == "STRUCTURAL")
if _struct.height:
    print("\nStructural failure cells (will be excluded from ANOVA):")
    for row in _struct.select(["model", "scenario", "run_id", "n_missing"]).iter_rows(named=True):
        print(f"  {row['model']}/{row['scenario']}/run{row['run_id']}: {row['n_missing']} missing")


In [ ]:
# Phase 5 — Cell 5.6: Classification verification

assert len(results_df) > 0, "No classifications produced"
assert set(results_df["predicted_label"].unique().to_list()).issubset(
    {"Positive", "Negative", "Neutral"}
), f"Invalid sentiment labels: {results_df['predicted_label'].unique().to_list()}"
assert set(results_df["predicted_aspect"].unique().to_list()).issubset(
    {"content", "presenter", "community", "audio_video", "meta", "unknown"}
), f"Invalid aspect labels: {results_df['predicted_aspect'].unique().to_list()}"
assert results_df["confidence"].is_between(0.0, 1.0).all(), "Confidence scores out of range"
assert results_df["latency_ms"].ge(0.0).all(), "Negative latency detected"
assert set(results_df["model_id"].unique().to_list()).issubset(
    set(MODELS.keys())
), f"Unknown model_id: {results_df['model_id'].unique().to_list()}"
assert set(results_df["scenario"].unique().to_list()).issubset(
    set(SCENARIOS)
), f"Unknown scenario: {results_df['scenario'].unique().to_list()}"

unique_cells = results_df.select("model_id", "scenario", "run_id").unique().shape[0]
if unique_cells < TOTAL_CELLS:
    print(f"[WARN] {TOTAL_CELLS - unique_cells} cell(s) missing from results_df "
          f"(got {unique_cells}/{TOTAL_CELLS}). Check coverage_report.csv for details.")
else:
    print(f"[OK] All {TOTAL_CELLS} cells present in results_df")

print(f"\u2705 Classification verification passed")
print(f"Total classified: {len(results_df):,}")
print(f"Cells: {unique_cells}/{TOTAL_CELLS}")
print(f"\nSentiment distribution:\n{results_df['predicted_label'].value_counts().sort('count', descending=True)}")
print(f"\nAspect distribution:\n{results_df['predicted_aspect'].value_counts().sort('count', descending=True)}")
print(f"\nMean latency: {results_df['latency_ms'].mean():.1f}ms")

In [ ]:
# Phase 5 — Cell 5.7: Generate Consensus Ground Truth (Tier 2 — "Silver GT")

consensus_pool_rids = set(consensus_row_ids)
consensus_results = results_df.filter(
    (pl.col("row_id").is_in(list(consensus_pool_rids))) & (pl.col("scenario") == "zero_shot")
)

consensus_rows: list[dict] = []
partial_rows: list[dict] = []
ambiguous_rows: list[dict] = []

for rid in sorted(consensus_pool_rids):
    rid_data = consensus_results.filter(pl.col("row_id") == rid)

    model_votes_label: dict[str, str] = {}
    model_votes_aspect: dict[str, str] = {}

    for model_slug in MODELS.keys():
        model_data = rid_data.filter(pl.col("model_id") == model_slug)
        if model_data.shape[0] == 0:
            continue

        labels = model_data["predicted_label"].to_list()
        modal_label = max(set(labels), key=labels.count)
        model_votes_label[model_slug] = modal_label

        aspects = model_data["predicted_aspect"].to_list()
        modal_aspect = max(set(aspects), key=aspects.count)
        model_votes_aspect[model_slug] = modal_aspect

    if len(model_votes_label) < 2:
        continue

    label_votes = list(model_votes_label.values())
    consensus_label = max(set(label_votes), key=label_votes.count)
    label_agreement = label_votes.count(consensus_label)

    aspect_votes = list(model_votes_aspect.values())
    consensus_aspect = max(set(aspect_votes), key=aspect_votes.count)
    aspect_agreement = aspect_votes.count(consensus_aspect)

    threshold = len(model_votes_label) * 2 / 3
    models_agreeing = [m for m, v in model_votes_label.items() if v == consensus_label]

    if label_agreement >= threshold and aspect_agreement >= threshold:
        consensus_rows.append({
            "row_id": rid,
            "consensus_label": consensus_label,
            "consensus_aspect": consensus_aspect,
            "label_agreement_count": label_agreement,
            "aspect_agreement_count": aspect_agreement,
            "models_agreeing": ",".join(sorted(models_agreeing)),
        })
    elif label_agreement >= threshold:
        partial_rows.append({
            "row_id": rid,
            "consensus_label": consensus_label,
            "label_agreement_count": label_agreement,
        })
    else:
        ambiguous_rows.append({
            "row_id": rid,
            **{f"{m}_label": v for m, v in model_votes_label.items()},
            **{f"{m}_aspect": v for m, v in model_votes_aspect.items()},
        })

gt_consensus = pl.DataFrame(consensus_rows) if consensus_rows else pl.DataFrame()
gt_partial = pl.DataFrame(partial_rows) if partial_rows else pl.DataFrame()
gt_ambiguous = pl.DataFrame(ambiguous_rows) if ambiguous_rows else pl.DataFrame()

if len(gt_consensus) > 0:
    gt_consensus.write_csv(str(GT_DIR / "ground_truth_consensus.csv"))
if len(gt_partial) > 0:
    gt_partial.write_csv(str(GT_DIR / "partial_consensus.csv"))
if len(gt_ambiguous) > 0:
    gt_ambiguous.write_csv(str(GT_DIR / "ambiguous_messages.csv"))

total_pool = len(consensus_pool_rids)
print(f"Consensus GT generation:")
print(f"  Full consensus (label+aspect): {len(consensus_rows)} / {total_pool} ({len(consensus_rows)/max(total_pool,1)*100:.1f}%)")
print(f"  Partial consensus (label only): {len(partial_rows)}")
print(f"  Ambiguous (no agreement): {len(ambiguous_rows)}")
if len(gt_consensus) > 0:
    print(f"\nConsensus label distribution:")
    print(gt_consensus["consensus_label"].value_counts().sort("count", descending=True))

---
## Phase 6: Metrics and Statistical Proof

**Goal:** Compute classification metrics against dual Ground Truth, prove statistical significance via two-way ANOVA, validate consensus proxy, demonstrate hallucination mitigation through variance analysis.

**Evaluation set:** 182 Human GT messages (primary) + ~300 Consensus GT (secondary)

In [ ]:
# Phase 6 — Cell 6.1: Load data and prepare for metrics

import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, cohen_kappa_score,
)

eval_gt_pd = eval_gt.to_pandas()
eval_gt_pd = eval_gt_pd.rename(columns={"label": "human_label", "aspect": "human_aspect"})

gt_consensus_pd = pd.DataFrame()
if (GT_DIR / "ground_truth_consensus.csv").exists():
    gt_consensus_pd = pl.read_csv(str(GT_DIR / "ground_truth_consensus.csv")).to_pandas()
    print(f"Consensus GT loaded: {len(gt_consensus_pd)} rows")

results_pd = results_df.to_pandas()

icl_holdout_ids = json.loads(holdout_path.read_text())
results_eval = results_pd[~results_pd["row_id"].isin(icl_holdout_ids)].copy()

print(f"Evaluation results: {len(results_eval):,} (excluding {len(icl_holdout_ids)} ICL holdout rows)")
print(f"Human GT: {len(eval_gt_pd)} rows")
print(f"Consensus GT: {len(gt_consensus_pd)} rows")
# ── Load coverage + consistency reports ───────────────────────────────────────
coverage_pd     = pd.read_csv(str(METRICS_DIR / "coverage_report.csv"))
consistency_pd  = pd.read_csv(str(METRICS_DIR / "consistency_report.csv"))

# STRUCTURAL_CELLS: (model, scenario) where any run exceeds the structural threshold
_STRUCTURAL_THRESHOLD = 50
_struct_mask    = coverage_pd["n_missing"] > _STRUCTURAL_THRESHOLD
STRUCTURAL_CELLS: set[tuple[str, str]] = set(
    zip(coverage_pd.loc[_struct_mask, "model"],
        coverage_pd.loc[_struct_mask, "scenario"])
)

# CONSISTENCY_TIERS: mean cross-run agreement per (model, scenario) → tier 1-4
_cons_mean = (
    consistency_pd
    .groupby(["model", "scenario"])["agreement_pct"]
    .mean()
    .reset_index(name="mean_agreement")
)

def _assign_tier(pct: float) -> int:
    if pct >= 0.90: return 1
    if pct >= 0.80: return 2
    if pct >= 0.70: return 3
    return 4

_cons_mean["consistency_tier"] = _cons_mean["mean_agreement"].apply(_assign_tier)

CONSISTENCY_TIERS: dict[tuple[str, str], dict] = {
    (row["model"], row["scenario"]): {
        "tier":           int(row["consistency_tier"]),
        "mean_agreement": float(row["mean_agreement"]),
    }
    for _, row in _cons_mean.iterrows()
}

print(f"\n=== Structural cells excluded from ANOVA ({len(STRUCTURAL_CELLS)} pairs) ===")
for m, s in sorted(STRUCTURAL_CELLS):
    print(f"  {m}/{s}")

print(f"\n=== Consistency tiers ===")
_tier_labels = {1: "Tier 1 (≥90%)", 2: "Tier 2 (80-89%)", 3: "Tier 3 (70-79%)", 4: "Tier 4 (<70%)"}
for tier in [1, 2, 3, 4]:
    members = [(m, s) for (m, s), v in CONSISTENCY_TIERS.items() if v["tier"] == tier]
    if members:
        print(f"  {_tier_labels[tier]}:")
        for m, s in sorted(members):
            pct = CONSISTENCY_TIERS[(m, s)]["mean_agreement"]
            print(f"    {m}/{s}: {pct:.1%}")


In [ ]:
# Phase 6 — Cell 6.2: Per-model-scenario metrics vs Human GT (primary)
#
# Primary metrics (accuracy, precision, recall, macro_f1, cohen_kappa) are
# computed from run_id=1 (modal representative). f1_mean_5runs and f1_std_5runs
# aggregate all 5 runs to show cross-run stability under T=0.3 stochasticity.
# Time: O(M * S * R * N). Space: O(M * S).

label_order = ["Negative", "Neutral", "Positive"]
metrics_human: list[dict] = []

for model_slug in MODELS.keys():
    for scenario in SCENARIOS:
        run_f1s: list[float] = []
        for run_id in range(1, NUM_RUNS + 1):
            run_data = results_eval[
                (results_eval["model_id"] == model_slug) &
                (results_eval["scenario"] == scenario) &
                (results_eval["run_id"] == run_id)
            ]
            merged = run_data.merge(eval_gt_pd[["row_id", "human_label"]], on="row_id", how="inner")
            if len(merged) == 0:
                continue
            f1 = f1_score(merged["human_label"], merged["predicted_label"], labels=label_order, average="macro", zero_division=0)
            run_f1s.append(f1)

        modal_run = results_eval[
            (results_eval["model_id"] == model_slug) &
            (results_eval["scenario"] == scenario) &
            (results_eval["run_id"] == 1)
        ]
        merged = modal_run.merge(eval_gt_pd[["row_id", "human_label"]], on="row_id", how="inner")
        if len(merged) == 0:
            continue

        y_true = merged["human_label"].values
        y_pred = merged["predicted_label"].values

        metrics_human.append({
            "model": model_slug,
            "scenario": scenario,
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_precision": round(precision_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0), 4),
            "macro_recall": round(recall_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0), 4),
            "macro_f1": round(f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0), 4),
            "cohen_kappa": round(cohen_kappa_score(y_true, y_pred), 4),
            "f1_mean_5runs": round(np.mean(run_f1s), 4) if run_f1s else 0.0,
            "f1_std_5runs": round(np.std(run_f1s, ddof=1), 4) if len(run_f1s) > 1 else 0.0,
        })

metrics_human_df = pd.DataFrame(metrics_human)
metrics_human_df.to_csv(str(METRICS_DIR / "metrics_vs_human_gt.csv"), index=False)

# ── Annotate with coverage + consistency ──────────────────────────────────────
_cov_agg = (
    coverage_pd
    .groupby(["model", "scenario"])["coverage_pct"]
    .mean()
    .reset_index(name="mean_coverage_pct")
)

metrics_human_df = (
    metrics_human_df
    .merge(_cov_agg, on=["model", "scenario"], how="left")
    .merge(
        _cons_mean[["model", "scenario", "mean_agreement", "consistency_tier"]],
        on=["model", "scenario"], how="left",
    )
)
metrics_human_df["is_structural"] = metrics_human_df.apply(
    lambda r: (r["model"], r["scenario"]) in STRUCTURAL_CELLS, axis=1
)

# Re-export with new columns
metrics_human_df.to_csv(str(METRICS_DIR / "metrics_vs_human_gt.csv"), index=False)
print(f"  Annotated +mean_coverage_pct, +mean_agreement, +consistency_tier, +is_structural")
print(f"  Structural rows flagged: {metrics_human_df['is_structural'].sum()}")
print("=== Metrics vs Human GT (Primary) ===")
print(metrics_human_df.sort_values("macro_f1", ascending=False).to_string(index=False))


In [ ]:
# Phase 6 — Cell 6.3: Per-model-scenario metrics vs Consensus GT (secondary)

metrics_consensus: list[dict] = []

if len(gt_consensus_pd) > 0:
    for model_slug in MODELS.keys():
        for scenario in SCENARIOS:
            modal_run = results_eval[
                (results_eval["model_id"] == model_slug) &
                (results_eval["scenario"] == scenario) &
                (results_eval["run_id"] == 1)
            ]
            merged = modal_run.merge(
                gt_consensus_pd[["row_id", "consensus_label"]],
                on="row_id", how="inner",
            )
            if len(merged) == 0:
                continue

            y_true = merged["consensus_label"].values
            y_pred = merged["predicted_label"].values

            metrics_consensus.append({
                "model": model_slug,
                "scenario": scenario,
                "accuracy": round(accuracy_score(y_true, y_pred), 4),
                "macro_precision": round(precision_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0), 4),
                "macro_recall": round(recall_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0), 4),
                "macro_f1": round(f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0), 4),
                "cohen_kappa": round(cohen_kappa_score(y_true, y_pred), 4),
            })

    metrics_consensus_df = pd.DataFrame(metrics_consensus)
    metrics_consensus_df.to_csv(str(METRICS_DIR / "metrics_vs_consensus_gt.csv"), index=False)
    print("=== Metrics vs Consensus GT (Secondary) ===")
    print(metrics_consensus_df.sort_values("macro_f1", ascending=False).to_string(index=False))
else:
    print("Consensus GT not available — skipping secondary metrics")


In [ ]:
# Phase 6 — Cell 6.4: Confusion matrices (vs Human GT)

for model_slug in MODELS.keys():
    for scenario in SCENARIOS:
        run_data = results_eval[
            (results_eval["model_id"] == model_slug) &
            (results_eval["scenario"] == scenario) &
            (results_eval["run_id"] == 1)
        ]
        merged = run_data.merge(eval_gt_pd[["row_id", "human_label"]], on="row_id", how="inner")
        if len(merged) == 0:
            continue
        cm = confusion_matrix(merged["human_label"], merged["predicted_label"], labels=label_order)
        cm_df = pd.DataFrame(cm, index=label_order, columns=label_order)
        cm_df.to_csv(str(METRICS_DIR / f"confusion_matrix_{model_slug}_{scenario}.csv"))

print(f"Exported {len(MODELS) * len(SCENARIOS)} confusion matrices")

In [ ]:
# Phase 6 — Cell 6.5: Aggregated run metrics for ANOVA (structural cells excluded)

aggregated_runs: list[dict] = []

for model_slug in MODELS.keys():
    for scenario in SCENARIOS:
        for run_id in range(1, NUM_RUNS + 1):
            run_data = results_eval[
                (results_eval["model_id"] == model_slug) &
                (results_eval["scenario"] == scenario) &
                (results_eval["run_id"] == run_id)
            ]
            merged = run_data.merge(eval_gt_pd[["row_id", "human_label"]], on="row_id", how="inner")
            if len(merged) == 0:
                continue

            y_true = merged["human_label"].values
            y_pred = merged["predicted_label"].values

            aggregated_runs.append({
                "model": model_slug,
                "scenario": scenario,
                "run_id": run_id,
                "accuracy": round(accuracy_score(y_true, y_pred), 4),
                "macro_f1": round(f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0), 4),
            })

agg_df = pd.DataFrame(aggregated_runs)

# Exclude structural-failure (model, scenario) pairs before ANOVA
_before = len(agg_df)
agg_df = agg_df[
    ~agg_df.apply(lambda r: (r["model"], r["scenario"]) in STRUCTURAL_CELLS, axis=1)
].reset_index(drop=True)
_excluded = _before - len(agg_df)

# _before excludes unrun models (empty merge → skipped by loop guard)
_expected_valid = _before - len(STRUCTURAL_CELLS) * NUM_RUNS
_valid_pairs    = _expected_valid // NUM_RUNS
assert len(agg_df) == _expected_valid, (
    f'Expected {_expected_valid} valid ANOVA rows '
    f'({_valid_pairs} pairs × {NUM_RUNS} runs), got {len(agg_df)}. '
    f'Excluded {_excluded} structural rows ({len(STRUCTURAL_CELLS)} pairs).'
)

agg_df.to_csv(str(METRICS_DIR / "aggregated_run_metrics.csv"), index=False)
print(f"[ANOVA] {len(agg_df)} valid rows ({_excluded} structural rows excluded)")
print(f"  Excluded pairs: {sorted(STRUCTURAL_CELLS)}")
print(agg_df.groupby(["model", "scenario"])["macro_f1"].agg(["mean", "std"]).round(4))


In [ ]:
# Phase 6 — Cell 6.6: Two-way ANOVA (model x scenario) with assumption checks
#
# Tests whether model choice and/or prompting scenario significantly affect
# macro F1 scores. Assumption checks are per interaction cell (model x scenario,
# n=5 each). Reports main effects, interaction term, and effect sizes (eta-squared).
# Eta-squared thresholds: small >= 0.01, medium >= 0.06, large >= 0.14.
# Falls back to Kruskal-Wallis + Mann-Whitney U if assumptions are violated.
# Time: O(C) where C = number of cells. Space: O(C).

import pingouin as pg

agg_df['cell'] = agg_df['model'] + '/' + agg_df['scenario']

normality = pg.normality(agg_df, dv='macro_f1', group='cell')
print('=== Shapiro-Wilk Normality Test (per model x scenario cell, n=5 each) ===')
print(normality)
normality_ok = normality['normal'].all()

homoscedasticity = pg.homoscedasticity(agg_df, dv='macro_f1', group='cell')
print('\n=== Levene\'s Test for Homoscedasticity (across all 9 cells) ===')
print(homoscedasticity)
levene_ok = homoscedasticity['equal_var'].values[0]

stat_tests: list[dict] = []

if normality_ok and levene_ok:
    print('\n\u2705 Assumptions met \u2014 running Two-Way ANOVA')
    anova_result = pg.anova(data=agg_df, dv='macro_f1', between=['model', 'scenario'], detailed=True)
    print(anova_result)

    for _, row in anova_result.iterrows():
        eta2 = round(row.get('np2', 0), 4)
        effect_label = 'large' if eta2 >= 0.14 else 'medium' if eta2 >= 0.06 else 'small' if eta2 >= 0.01 else 'negligible'
        stat_tests.append({
            'test': 'two_way_anova',
            'source': row['Source'],
            'F': round(row.get('F', 0), 4),
            'p_value': round(row.get('p_unc', 1), 6),
            'eta_squared': eta2,
            'effect_size': effect_label,
        })

    interaction_row = anova_result[anova_result['Source'] == 'model * scenario']
    if len(interaction_row) > 0:
        int_p = interaction_row['p_unc'].values[0]
        int_eta2 = interaction_row['np2'].values[0]
        print(f'\nInteraction (model x scenario): p={int_p:.6f}, \u03b7\u00b2={int_eta2:.4f}')
        if int_p < 0.05:
            print('\u26a0 Significant interaction \u2014 interpret main effects with caution')

    print('\n=== Pairwise t-tests (Bonferroni) ===')
    pairwise_model = pg.pairwise_tests(data=agg_df, dv='macro_f1', between='model', padjust='bonf')
    print(pairwise_model)
    pairwise_scenario = pg.pairwise_tests(data=agg_df, dv='macro_f1', between='scenario', padjust='bonf')
    print(pairwise_scenario)
else:
    print('\n\u26a0 Assumptions violated \u2014 running non-parametric Kruskal-Wallis')
    kw_model = pg.kruskal(data=agg_df, dv='macro_f1', between='model')
    kw_scenario = pg.kruskal(data=agg_df, dv='macro_f1', between='scenario')
    print('Kruskal-Wallis (model):')
    print(kw_model)
    print('Kruskal-Wallis (scenario):')
    print(kw_scenario)

    stat_tests.append({'test': 'kruskal_wallis', 'source': 'model', 'H': round(kw_model['H'].values[0], 4), 'p_value': round(kw_model['p_unc'].values[0], 6)})
    stat_tests.append({'test': 'kruskal_wallis', 'source': 'scenario', 'H': round(kw_scenario['H'].values[0], 4), 'p_value': round(kw_scenario['p_unc'].values[0], 6)})

    pairwise_model = pg.pairwise_tests(data=agg_df, dv='macro_f1', between='model', parametric=False, padjust='bonf')
    pairwise_scenario = pg.pairwise_tests(data=agg_df, dv='macro_f1', between='scenario', parametric=False, padjust='bonf')
    print(pairwise_model)
    print(pairwise_scenario)

pd.DataFrame(stat_tests).to_csv(str(METRICS_DIR / 'statistical_tests.csv'), index=False)
print(f'\nStatistical tests exported to {METRICS_DIR / "statistical_tests.csv"}')


In [ ]:
# Phase 6 — Cell 6.7: Cross-model agreement (Fleiss' kappa + pairwise Cohen's kappa)

from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

agreement_results: list[dict] = []

zs_run1 = results_eval[
    (results_eval["scenario"] == "zero_shot") & (results_eval["run_id"] == 1)
]

# Use only models that actually have data (excludes unrun models like llama3.1-8b, aya-expanse-8b)
model_list = sorted(zs_run1["model_id"].unique())

common_rids = set(zs_run1["row_id"].unique())
for model_slug in model_list:
    model_rids = set(zs_run1[zs_run1["model_id"] == model_slug]["row_id"].unique())
    common_rids &= model_rids

print(f"Models in zero_shot/run1: {len(model_list)} → {model_list}")
print(f"Common row_ids across all models: {len(common_rids)}")

rating_table: list[list[str]] = []

for rid in sorted(common_rids):
    row_ratings: list[str] = []
    for model_slug in model_list:
        pred = zs_run1[
            (zs_run1["model_id"] == model_slug) & (zs_run1["row_id"] == rid)
        ]["predicted_label"].values
        if len(pred) > 0:
            row_ratings.append(pred[0])
    if len(row_ratings) == len(model_list):
        rating_table.append(row_ratings)

label_to_int = {"Negative": 0, "Neutral": 1, "Positive": 2}
int_table = [[label_to_int[r] for r in row] for row in rating_table]
fleiss_table, _ = aggregate_raters(np.array(int_table))
fleiss_k = fleiss_kappa(fleiss_table)

agreement_results.append({"metric": "fleiss_kappa", "value": round(fleiss_k, 4), "models": "all"})
print(f"Fleiss' kappa ({len(model_list)} models): {fleiss_k:.4f}")

for i, m1 in enumerate(model_list):
    for m2 in model_list[i + 1:]:
        m1_labels = []
        m2_labels = []
        for rid in sorted(common_rids):
            p1 = zs_run1[(zs_run1["model_id"] == m1) & (zs_run1["row_id"] == rid)]["predicted_label"].values
            p2 = zs_run1[(zs_run1["model_id"] == m2) & (zs_run1["row_id"] == rid)]["predicted_label"].values
            if len(p1) > 0 and len(p2) > 0:
                m1_labels.append(p1[0])
                m2_labels.append(p2[0])

        if m1_labels:
            ck = cohen_kappa_score(m1_labels, m2_labels)
            agreement_results.append({"metric": "cohen_kappa", "value": round(ck, 4), "models": f"{m1} vs {m2}"})
            print(f"Cohen's kappa ({m1} vs {m2}): {ck:.4f}")

agreement_df = pd.DataFrame(agreement_results)
agreement_df.to_csv(str(METRICS_DIR / "cross_model_agreement.csv"), index=False)

In [ ]:
# Phase 6 — Cell 6.8: Variance analysis + per-message entropy

from scipy.stats import entropy as shannon_entropy

variance_report: dict = {"per_model_scenario": {}, "within_model": {}, "between_model": {}}

for model_slug in MODELS.keys():
    for scenario in SCENARIOS:
        f1s = agg_df[(agg_df["model"] == model_slug) & (agg_df["scenario"] == scenario)]["macro_f1"].values
        if len(f1s) > 1:
            sigma_sq = float(np.var(f1s, ddof=1))
            variance_report["per_model_scenario"][f"{model_slug}_{scenario}"] = {
                "sigma_squared": round(sigma_sq, 6),
                "std": round(np.std(f1s, ddof=1), 4),
                "mean_f1": round(np.mean(f1s), 4),
            }

for model_slug in MODELS.keys():
    zs_f1s = agg_df[(agg_df["model"] == model_slug) & (agg_df["scenario"] == "zero_shot")]["macro_f1"].values
    icl_f1s = agg_df[(agg_df["model"] == model_slug) & (agg_df["scenario"] == "icl")]["macro_f1"].values
    if len(zs_f1s) > 1 and len(icl_f1s) > 1:
        variance_report["within_model"][model_slug] = {
            "zero_shot_var": round(float(np.var(zs_f1s, ddof=1)), 6),
            "icl_var": round(float(np.var(icl_f1s, ddof=1)), 6),
            "icl_reduces_variance": bool(np.var(icl_f1s, ddof=1) < np.var(zs_f1s, ddof=1)),
        }

for scenario in SCENARIOS:
    model_means = {}
    for model_slug in MODELS.keys():
        f1s = agg_df[(agg_df["model"] == model_slug) & (agg_df["scenario"] == scenario)]["macro_f1"].values
        if len(f1s) > 0:
            model_means[model_slug] = float(np.mean(f1s))
    if model_means:
        between_var = float(np.var(list(model_means.values()), ddof=1)) if len(model_means) > 1 else 0.0
        variance_report["between_model"][scenario] = {
            "model_means": {k: round(v, 4) for k, v in model_means.items()},
            "between_model_var": round(between_var, 6),
        }

Path(str(METRICS_DIR / "variance_report.json")).write_text(json.dumps(variance_report, indent=2))
print("=== Variance Report ===")
print(json.dumps(variance_report, indent=2))

entropy_rows: list[dict] = []
eval_rids_in_results = set(zs_run1["row_id"].unique())

for rid in sorted(eval_rids_in_results):
    rid_labels = zs_run1[zs_run1["row_id"] == rid]["predicted_label"].values
    if len(rid_labels) == 0:
        continue
    label_counts = pd.Series(rid_labels).value_counts(normalize=True)
    probs = [label_counts.get(l, 0) for l in label_order]
    ent = float(shannon_entropy(probs, base=2)) if sum(probs) > 0 else 0.0
    modal = label_counts.idxmax()
    agreement = int(pd.Series(rid_labels).value_counts().max())
    entropy_rows.append({
        "row_id": int(rid),
        "entropy": round(ent, 4),
        "modal_label": modal,
        "agreement_count": agreement,
    })

entropy_df = pd.DataFrame(entropy_rows)
entropy_df.to_csv(str(METRICS_DIR / "per_message_entropy.csv"), index=False)
print(f"\nPer-message entropy: {len(entropy_df)} messages")
print(f"Mean entropy: {entropy_df['entropy'].mean():.4f}")
print(f"High-entropy messages (>1.0): {(entropy_df['entropy'] > 1.0).sum()}")

In [ ]:
# Phase 6 — Cell 6.9: Proxy validation (Consensus GT vs Human GT)

from scipy.stats import pearsonr

if len(metrics_consensus) > 0 and len(metrics_human) > 0:
    human_f1s = metrics_human_df.set_index(["model", "scenario"])["macro_f1"]
    consensus_f1s = metrics_consensus_df.set_index(["model", "scenario"])["macro_f1"]
    common_idx = human_f1s.index.intersection(consensus_f1s.index)

    if len(common_idx) >= 3:
        h_vals = human_f1s.loc[common_idx].values
        c_vals = consensus_f1s.loc[common_idx].values
        pearson_r, pearson_p = pearsonr(h_vals, c_vals)

        proxy_result = {
            "pearson_r": round(pearson_r, 4),
            "pearson_p": round(pearson_p, 6),
            "n_comparisons": len(common_idx),
            "proxy_validated": bool(pearson_r >= 0.85),
        }
        pd.DataFrame([proxy_result]).to_csv(str(METRICS_DIR / "proxy_validation.csv"), index=False)
        print(f"Proxy validation: Pearson r = {pearson_r:.4f} (p = {pearson_p:.6f})")
        print(f"Proxy {'\u2705 VALIDATED' if pearson_r >= 0.85 else '\u274c NOT validated'} (threshold: r >= 0.85)")
    else:
        print(f"Not enough common configurations for proxy validation ({len(common_idx)} < 3)")
else:
    print("Proxy validation skipped — consensus or human metrics not available")

In [ ]:
# Phase 6 — Cell 6.9b: Consensus GT spot-check (T4.9)
# Display 50 random consensus GT messages for manual verification

if len(gt_consensus_pd) >= 50:
    spot_check = gt_consensus_pd.sample(n=50, random_state=random_seed)
else:
    spot_check = gt_consensus_pd.copy()

if len(spot_check) > 0:
    spot_with_text = spot_check.merge(
        cleaned.select(["row_id", "message"]).to_pandas(),
        on="row_id", how="inner",
    )

    print(f"=== Consensus GT Spot-Check ({len(spot_with_text)} messages) ===")
    print("Review these labels — do they match your judgment?\n")

    # Use actual model count from data, not MODELS dict (which includes unrun models)
    _n_models_ran = results_eval["model_id"].nunique()

    for _, row in spot_with_text.head(50).iterrows():
        text_preview = str(row["message"])[:80]
        print(
            f"  [{row['consensus_label']:>8s} / {row['consensus_aspect']:<12s}] "
            f"(agree: {row['label_agreement_count']}/{_n_models_ran}) "
            f"{text_preview}"
        )

    unanimous = (spot_with_text["label_agreement_count"] == _n_models_ran).sum()
    majority = (spot_with_text["label_agreement_count"] > _n_models_ran / 2).sum()
    print(f"\nAgreement breakdown in sample: unanimous={unanimous}, majority={majority}")
    print("If >80% look correct, the consensus proxy is reliable for secondary evaluation.")
else:
    print("No consensus GT available for spot-check")

In [ ]:
# Phase 6 — Cell 6.10: Aspect distribution + per-aspect F1
#
# Computes predicted aspect counts and per-aspect F1 scores for each
# (model, scenario) pair. F1 values are stored in a separate dict keyed
# by (model, scenario, aspect) to avoid fragile negative-index assignment.
# Time: O(M * S * A * N). Space: O(M * S * A).

aspect_order = ['content', 'presenter', 'community', 'audio_video', 'meta', 'unknown']

aspect_dist_rows: list[dict] = []
f1_lookup: dict[tuple[str, str, str], float] = {}

for model_slug in MODELS.keys():
    for scenario in SCENARIOS:
        run_data = results_eval[
            (results_eval['model_id'] == model_slug) &
            (results_eval['scenario'] == scenario) &
            (results_eval['run_id'] == 1)
        ]
        if len(run_data) == 0:
            continue
        counts = run_data['predicted_aspect'].value_counts()
        for aspect in aspect_order:
            aspect_dist_rows.append({
                'model': model_slug,
                'scenario': scenario,
                'aspect': aspect,
                'count': int(counts.get(aspect, 0)),
            })

        merged = run_data.merge(eval_gt_pd[['row_id', 'human_aspect']], on='row_id', how='inner')
        if len(merged) > 0:
            per_aspect_f1 = f1_score(
                merged['human_aspect'], merged['predicted_aspect'],
                labels=aspect_order, average=None, zero_division=0,
            )
            for aspect, f1_val in zip(aspect_order, per_aspect_f1):
                f1_lookup[(model_slug, scenario, aspect)] = round(f1_val, 4)

for row in aspect_dist_rows:
    row['f1_vs_human'] = f1_lookup.get((row['model'], row['scenario'], row['aspect']), None)

aspect_dist_df = pd.DataFrame(aspect_dist_rows)
aspect_dist_df.to_csv(str(METRICS_DIR / 'aspect_distribution.csv'), index=False)
print('Aspect distribution and per-aspect F1 exported')
print(aspect_dist_df.head(15))


In [ ]:
# Phase 6 — Cell 6.11: Latency table

latency_rows: list[dict] = []
for model_slug in MODELS.keys():
    for scenario in SCENARIOS:
        model_scenario_data = results_pd[
            (results_pd["model_id"] == model_slug) & (results_pd["scenario"] == scenario)
        ]
        if len(model_scenario_data) == 0:
            continue
        f1_row = metrics_human_df[
            (metrics_human_df["model"] == model_slug) & (metrics_human_df["scenario"] == scenario)
        ]
        mean_f1 = float(f1_row["macro_f1"].values[0]) if len(f1_row) > 0 else 0.0

        latency_rows.append({
            "model": model_slug,
            "scenario": scenario,
            "mean_latency_ms": round(model_scenario_data["latency_ms"].mean(), 2),
            "total_latency_ms": round(model_scenario_data["latency_ms"].sum(), 2),
            "classifications": len(model_scenario_data),
            "macro_f1": mean_f1,
            "cost_usd": 0.0,
        })

latency_df = pd.DataFrame(latency_rows)
latency_df.to_csv(str(METRICS_DIR / "latency_table.csv"), index=False)
print("=== Latency Table ===")
print(latency_df.to_string(index=False))

In [ ]:
# Phase 6 — Cell 6.12: Phase 6 verification + export summary

required_files = [
    "metrics_vs_human_gt.csv",
    "aggregated_run_metrics.csv",
    "statistical_tests.csv",
    "cross_model_agreement.csv",
    "per_message_entropy.csv",
    "variance_report.json",
    "aspect_distribution.csv",
    "latency_table.csv",
    "coverage_report.csv",
    "consistency_report.csv",
]

optional_files = [
    "metrics_vs_consensus_gt.csv",
    "proxy_validation.csv",
]

# Use only models that actually ran (excludes unrun models like llama3.1-8b, aya-expanse-8b)
_ran_models = sorted(results_eval["model_id"].unique())
cm_files = [
    f"confusion_matrix_{m}_{s}.csv" for m in _ran_models for s in SCENARIOS
]

missing = []
for f in required_files + cm_files:
    if not (METRICS_DIR / f).exists():
        missing.append(f)

if missing:
    print(f"\u274c Missing required files: {missing}")
else:
    print(f"\u2705 All {len(required_files) + len(cm_files)} required metric files present")

for f in optional_files:
    status = "\u2705" if (METRICS_DIR / f).exists() else "\u23f3 (pending)"
    print(f"  {status} {f}")

print(f"\n=== Best Configuration ===")
best = metrics_human_df.sort_values("macro_f1", ascending=False).iloc[0]
print(f"  Model: {best['model']}")
print(f"  Scenario: {best['scenario']}")
print(f"  Macro F1: {best['macro_f1']:.4f}")
print(f"  Accuracy: {best['accuracy']:.4f}")

---
## Phase 7: Visualization

**Goal:** Generate thesis-ready figures — confusion matrices, variance scatter, metrics bars, model comparison heatmap, interactive sentiment timeline.

In [ ]:
# Phase 7 — Cell 7.1: Visualization imports and theme

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

MODEL_COLORS = {
    "qwen3.5-4b":       "#1f77b4",  # blue
    "phi4-mini":        "#ff7f0e",  # orange
    "deepseek-r1-7b":   "#2ca02c",  # green
    "qwen2.5-3b":       "#d62728",  # red
    "llama3.2-3b":      "#9467bd",  # purple
    "deepseek-r1-1.5b": "#8c564b",  # brown
    "gemma2-2b":        "#e377c2",  # pink
    "qwen2.5-7b":       "#17becf",  # cyan
    "llama3.1-8b":      "#aec7e8",  # light blue
    "aya-expanse-8b":   "#7f7f7f",  # gray
    "gemma2-9b":        "#bcbd22",  # yellow-green
}
SCENARIO_MARKERS = {"zero_shot": "o", "icl": "s", "cot": "D"}

figure_catalog: list[dict] = []

def save_figure(fig, name: str, formats: tuple[str, ...] = ("png", "svg")):
    """Save a matplotlib figure in multiple formats and register in catalog.

    @param fig: Matplotlib Figure object.
    @param name: Base filename without extension.
    @param formats: Tuple of file formats to export.
    Time: O(1). Space: O(pixels).
    """
    for fmt in formats:
        path = VIZ_DIR / f"{name}.{fmt}"
        fig.savefig(str(path), format=fmt, bbox_inches="tight")
    figure_catalog.append({"name": name, "formats": ",".join(formats), "path": str(VIZ_DIR / name)})
    plt.close(fig)

print("Visualization setup complete")

In [ ]:
# Phase 7 — Cell 7.2: Confusion matrix heatmaps (9 total)

for model_slug in MODELS.keys():
    for scenario in SCENARIOS:
        cm_path = METRICS_DIR / f"confusion_matrix_{model_slug}_{scenario}.csv"
        if not cm_path.exists():
            continue
        cm_data = pd.read_csv(str(cm_path), index_col=0)

        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(
            cm_data, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_order, yticklabels=label_order, ax=ax,
        )
        ax.set_title(f"{model_slug} / {scenario}")
        ax.set_ylabel("True Label")
        ax.set_xlabel("Predicted Label")
        save_figure(fig, f"confusion_matrix_{model_slug}_{scenario}")

print(f"Exported {len(MODELS) * len(SCENARIOS)} confusion matrix heatmaps")

In [ ]:
# Phase 7 — Cell 7.3: Model x Scenario heatmap (F1 scores)

heatmap_data = metrics_human_df.pivot(index="model", columns="scenario", values="macro_f1")
heatmap_data = heatmap_data[SCENARIOS]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heatmap_data, annot=True, fmt=".3f", cmap="YlOrRd",
    linewidths=1, ax=ax, vmin=0, vmax=1,
)
ax.set_title("Macro F1 Score: Model x Scenario")
ax.set_ylabel("Model")
ax.set_xlabel("Scenario")
save_figure(fig, "model_scenario_heatmap")
print("Model x Scenario heatmap exported")

In [ ]:
# Phase 7 — Cell 7.3b: Cross-run consistency heatmap (model × scenario)
#
# Mean cross-run agreement (%) per (model, scenario).
# Structural failure cells shown in grey with 'N/A'.
# Color scale: RdYlGn (red=low consistency, green=high).

import numpy as np

_cons_pivot = _cons_mean.pivot(index="model", columns="scenario", values="mean_agreement")
_cons_pivot = _cons_pivot.reindex(columns=SCENARIOS)   # preserve scenario order

# Annotation matrix: pct string for valid cells, 'N/A' for structural
_annot = _cons_pivot.copy().astype(object)
for col in _annot.columns:
    for idx in _annot.index:
        val = _annot.loc[idx, col]
        if (idx, col) in STRUCTURAL_CELLS:
            _annot.loc[idx, col] = "N/A"
        elif isinstance(val, float) and not np.isnan(val):
            _annot.loc[idx, col] = f"{val:.1%}"

# Numeric matrix: NaN out structural cells so colormap skips them
_heat_vals = _cons_pivot.copy().astype(float)
for m, s in STRUCTURAL_CELLS:
    if m in _heat_vals.index and s in _heat_vals.columns:
        _heat_vals.loc[m, s] = np.nan

_is_na = _heat_vals.isna()

fig, ax = plt.subplots(figsize=(8, 6))

# Valid cells — coloured heatmap
sns.heatmap(
    _heat_vals, annot=_annot, fmt="", cmap="RdYlGn",
    linewidths=1, ax=ax, vmin=0.4, vmax=1.0,
    cbar_kws={"label": "Mean cross-run agreement"},
    mask=_is_na,
)

# Structural cells — grey overlay with 'N/A'
if _is_na.any().any():
    _na_annot = _annot.where(_is_na, other="")
    sns.heatmap(
        _is_na.astype(float), annot=_na_annot, fmt="",
        cmap=["#cccccc"], linewidths=1, ax=ax,
        cbar=False, mask=~_is_na,
    )

ax.set_title("Cross-Run Consistency: Model × Scenario\n"
             "(mean run1 vs run2–5 label agreement)")
ax.set_ylabel("Model")
ax.set_xlabel("Scenario")
save_figure(fig, "consistency_heatmap")
print("Consistency heatmap exported")

# Print tier summary alongside
print("\nTier summary (Tier 1=≥90%, 2=80-89%, 3=70-79%, 4=<70%):")
for tier in [1, 2, 3, 4]:
    members = [(m, s, CONSISTENCY_TIERS[(m, s)]['mean_agreement'])
               for (m, s) in CONSISTENCY_TIERS if CONSISTENCY_TIERS[(m, s)]['tier'] == tier]
    for m, s, pct in sorted(members):
        structural_note = " [STRUCTURAL]" if (m, s) in STRUCTURAL_CELLS else ""
        print(f"  Tier {tier}  {m}/{s}: {pct:.1%}{structural_note}")

In [ ]:
# Phase 7 — Cell 7.4: Metrics comparison bar chart with error bars

fig, ax = plt.subplots(figsize=(12, 6))
bar_data = metrics_human_df.copy()

# Use only models present in data (excludes unrun llama3.1-8b, aya-expanse-8b)
_ran_models = sorted(bar_data["model"].unique())

x_positions = np.arange(len(SCENARIOS))
bar_width = 0.8 / len(_ran_models)

for i, model_slug in enumerate(_ran_models):
    model_data = bar_data[bar_data["model"] == model_slug].set_index("scenario").reindex(SCENARIOS)
    means = model_data["f1_mean_5runs"].values
    stds = model_data["f1_std_5runs"].fillna(0).values
    ax.bar(
        x_positions + i * bar_width, means, bar_width,
        yerr=stds, capsize=4, label=model_slug,
        color=MODEL_COLORS.get(model_slug, "#888888"), alpha=0.85,
    )

ax.set_xlabel("Prompting Strategy")
ax.set_ylabel("Macro F1 (mean \u00b1 std across 5 runs)")
ax.set_title("Classification Performance: Model x Strategy")
ax.set_xticks(x_positions + bar_width * (len(_ran_models) - 1) / 2)
ax.set_xticklabels([s.replace("_", " ").title() for s in SCENARIOS])
ax.legend(title="Model", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_figure(fig, "metrics_comparison")
print("Metrics comparison bar chart exported")

In [ ]:
# Phase 7 — Cell 7.5: Variance / Entropy scatter

if len(entropy_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    scatter = ax.scatter(
        entropy_df["row_id"], entropy_df["entropy"],
        c=entropy_df["modal_label"].map({"Positive": "green", "Negative": "red", "Neutral": "gray"}),
        alpha=0.6, s=20, edgecolors="none",
    )
    ax.set_xlabel("Row ID (temporal order)")
    ax.set_ylabel("Shannon Entropy (bits)")
    ax.set_title("Per-Message Classification Entropy Across 3 Models")
    ax.axhline(y=1.0, color="red", linestyle="--", alpha=0.5, label="High disagreement threshold")
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="green", label="Positive"),
        Patch(facecolor="red", label="Negative"),
        Patch(facecolor="gray", label="Neutral"),
    ]
    ax.legend(handles=legend_elements, title="Modal Label")
    save_figure(fig, "variance_scatter")
    print("Variance/entropy scatter exported")

In [ ]:
# Phase 7 — Cell 7.6: Latency vs F1 scatter (efficiency frontier)

if len(latency_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 7))
    for _, row in latency_df.iterrows():
        ax.scatter(
            row["mean_latency_ms"], row["macro_f1"],
            c=MODEL_COLORS.get(row["model"], "gray"),
            marker=SCENARIO_MARKERS.get(row["scenario"], "o"),
            s=120, zorder=5,
        )
        ax.annotate(
            f"{row['model']}\n{row['scenario']}",
            (row["mean_latency_ms"], row["macro_f1"]),
            fontsize=7, ha="center", va="bottom", xytext=(0, 8),
            textcoords="offset points",
        )

    ax.set_xlabel("Mean Latency (ms/message)")
    ax.set_ylabel("Macro F1")
    ax.set_title("Efficiency Frontier: Latency vs Classification Quality")
    ax.grid(alpha=0.3)
    from matplotlib.lines import Line2D
    model_handles = [Line2D([0], [0], marker="o", color=c, linestyle="", markersize=8, label=m) for m, c in MODEL_COLORS.items()]
    scenario_handles = [Line2D([0], [0], marker=mk, color="gray", linestyle="", markersize=8, label=s) for s, mk in SCENARIO_MARKERS.items()]
    ax.legend(handles=model_handles + scenario_handles, loc="lower right", fontsize=8)
    save_figure(fig, "latency_f1_scatter")
    print("Latency vs F1 scatter exported")

In [ ]:
# Phase 7 — Cell 7.7: Interactive sentiment timeline (full dataset)

best_model = best["model"]
best_scenario = best["scenario"]

print(f"Best configuration: {best_model} / {best_scenario}")
print("Loading best model for full-dataset classification...")

full_classify_needed = not (CLASSIFIER_DIR / "full_classified.parquet").exists()

if full_classify_needed:
    clear_vram()
    best_hf_id = MODELS[best_model]

    if USE_VLLM:
        hf_token = None
        llm = LLM(
            model=best_hf_id, dtype=VLLM_DTYPE, enforce_eager=ENFORCE_EAGER,
            max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL, hf_token=hf_token,
        )
        structured_params = StructuredOutputsParams(json=LLMOutput.model_json_schema())
        sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=512, structured_outputs=structured_params)
    else:
        ollama_tag = OLLAMA_MODELS[best_model]
        print(f"[OLLAMA] Using {ollama_tag} for full-dataset classification")
        llm = None
        sampling = None

    if best_scenario == "icl":
        prompt_builder = create_icl_prompt_builder(holdout_messages, run_id=1)
    else:
        prompt_builder = STRATEGY_REGISTRY[best_scenario]

    all_texts = cleaned["message"].to_list()
    all_rids = cleaned["row_id"].to_list()
    all_times = cleaned["time_in_seconds"].to_list()

    prompts = [prompt_builder(text, rid) for text, rid in zip(all_texts, all_rids)]
    print(f"Classifying {len(prompts):,} messages...")

    full_results: list[dict] = []
    if USE_VLLM:
        batch_start = time.perf_counter() * 1000
        outputs = llm.generate(prompts, sampling)
        batch_elapsed = time.perf_counter() * 1000 - batch_start
        per_msg_latency = batch_elapsed / len(prompts)
        _ollama_outputs = [(out.outputs[0].text, per_msg_latency) for out in outputs]
    else:
        _ollama_outputs = []
        for prompt in prompts:
            t0 = time.perf_counter() * 1000
            try:
                raw = ollama_classify(prompt, OLLAMA_MODELS[best_model])
            except Exception as e:
                _ollama_outputs.append((None, 0))
                continue
            _ollama_outputs.append((raw, time.perf_counter() * 1000 - t0))
    _partial_path = CLASSIFIER_DIR / "full_classified_partial.parquet"
    for idx, ((raw_text, latency), rid, t) in enumerate(zip(_ollama_outputs, all_rids, all_times)):
        if raw_text is None:
            continue
        parsed = safe_parse_output(raw_text, rid, best_model, best_scenario, 1, latency)
        if parsed:
            d = parsed.model_dump()
            d["predicted_label"] = d.pop("label")
            d["predicted_aspect"] = d.pop("aspect")
            d["time_in_seconds"] = t
            full_results.append(d)
        if (idx + 1) % 500 == 0:
            pl.DataFrame(full_results).write_parquet(str(_partial_path))
            print(f"  [{idx + 1}/{len(_ollama_outputs)}] partial checkpoint saved", flush=True)

    full_df = pl.DataFrame(full_results)
    full_df.write_parquet(str(CLASSIFIER_DIR / "full_classified.parquet"))
    _partial_path.unlink(missing_ok=True)
    print(f"Full classification: {len(full_df):,} messages")

    del llm
    clear_vram()
else:
    full_df = pl.read_parquet(str(CLASSIFIER_DIR / "full_classified.parquet"))
    print(f"Loaded existing full classification: {len(full_df):,} messages")

full_pd = full_df.to_pandas()

sentiment_colors = {"Positive": "#2ecc71", "Negative": "#e74c3c", "Neutral": "#95a5a6"}

fig = go.Figure()
for label_val in ["Positive", "Negative", "Neutral"]:
    subset = full_pd[full_pd["predicted_label"] == label_val]
    fig.add_trace(go.Scattergl(
        x=subset["time_in_seconds"] / 60,
        y=[label_val] * len(subset),
        mode="markers",
        marker=dict(size=3, color=sentiment_colors[label_val], opacity=0.5),
        name=label_val,
        hovertext=subset["reasoning"].str[:100] if "reasoning" in subset.columns else None,
    ))

window_minutes = 5
full_pd["time_bin"] = (full_pd["time_in_seconds"] // (window_minutes * 60)) * window_minutes
rolling = full_pd.groupby("time_bin").apply(
    lambda g: pd.Series({"positive_ratio": (g["predicted_label"] == "Positive").mean()})
).reset_index()

fig.add_trace(go.Scatter(
    x=rolling["time_bin"],
    y=rolling["positive_ratio"],
    mode="lines",
    name=f"Positive ratio ({window_minutes}min window)",
    yaxis="y2",
    line=dict(color="#3498db", width=2),
))

fig.update_layout(
    title=f"Sentiment Timeline — {len(videos)} video(s), {len(full_pd):,} messages",
    xaxis_title="Time (minutes)",
    yaxis_title="Sentiment",
    yaxis2=dict(title="Positive Ratio", overlaying="y", side="right", range=[0, 1]),
    height=600,
    xaxis=dict(rangeslider=dict(visible=True)),
    template="plotly_white",
)

timeline_path = VIZ_DIR / "sentiment_timeline.html"
fig.write_html(str(timeline_path))
figure_catalog.append({"name": "sentiment_timeline", "formats": "html", "path": str(timeline_path)})

try:
    fig.write_image(str(VIZ_DIR / "sentiment_timeline.png"), width=1400, height=600, scale=2)
    figure_catalog.append({"name": "sentiment_timeline_static", "formats": "png", "path": str(VIZ_DIR / "sentiment_timeline.png")})
except Exception as export_error:
    print(f"Static export failed (Kaleido): {export_error}")

print(f"Timeline exported: {timeline_path}")
fig.show()


In [ ]:
# Phase 7 — Cell 7.8: Export figure catalog + final summary

pd.DataFrame(figure_catalog).to_csv(str(VIZ_DIR / "figure_catalog.csv"), index=False)
print(f"Figure catalog: {len(figure_catalog)} entries -> {VIZ_DIR / 'figure_catalog.csv'}")

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)

_total_raw = sum(v["raw_chat"].shape[0] for v in videos)
print(f"\nVideos: {len(videos)}")
print(f"Messages: {_total_raw:,} raw -> {cleaned.shape[0]:,} cleaned")
print(f"GT: {gt_human.shape[0]} human + {len(consensus_rows)} consensus")
print(f"Runs: {TOTAL_CELLS} classification cells")

print(f"\n=== Best Configuration ===")
print(f"  {best['model']} / {best['scenario']}: F1={best['macro_f1']:.4f}")

print(f"\n=== Output Directories ===")
for name, path in [("Cleaned", CLEANED_DIR), ("Ground Truth", GT_DIR), ("Classifier", CLASSIFIER_DIR), ("Metrics", METRICS_DIR), ("Visualizations", VIZ_DIR)]:
    file_count = len(list(path.glob("*")))
    print(f"  {name}: {path} ({file_count} files)")